# Landsat 1987-2026 cho geoportal Hải Phòng: chuẩn hoá, PCA, embedding, đẩy lên Hugging Face

Notebook này chạy trên **tài khoản chính** (không cần Earth Engine), runtime **A100**. Ảnh do các tài khoản GEE phụ dựng
(notebook `ls_GEE_tkX`) được gom về đây. Mọi bước đều chạy tiếp được: bước đã có kết quả hợp lệ thì bỏ qua.

## Hai phương án dữ liệu

| | A: ảnh có sẵn trên Drive (`HP-YYYY-Mosaic.tif`) | **B: dựng lại trên GEE (khuyến nghị)** |
|---|---|---|
| vùng | Đông Hải Phòng (ranh giới cũ) | Hải Phòng mới (cùng khung với S2) |
| mùa | gộp mọi tháng trong năm: nhiệt độ bề mặt trung vị dao động mạnh giữa các năm, dấu hiệu trộn mùa | mùa khô 1/11 năm trước đến 30/4, như ảnh S2 của geoportal |
| lọc | mây, bóng mây | thêm: bão hoà, phản xạ ngoài khoảng, khói mù (độ mờ khí quyển TM/ETM+, aerosol OLI), mép cảnh và khe SLC-off |
| cảm biến | TM, OLI (không có ETM+) | TM, ETM+, OLI, OLI-2 theo chính sách từng năm, Landsat 7 bỏ từ 2017 |
| năm thiếu | 2012 | mùa khô 2012 dựng từ ETM+ SLC-off (có cờ) |
| truy vết | không | băng NGUON (cách lấy điểm ảnh), NOBS, CAMBIEN (cảm biến đã góp) |

Cả hai đều qua cùng một chuỗi sau đó: lưới 30 m trùng khít 3 × 3 điểm ảnh S2, **chuẩn hoá tương đối IR-MAD** về một ảnh
tham chiếu chung (OLI mùa khô 2015-2020) có cổng chất lượng, cầu nối với S2 (chỉ chẩn đoán), COG, PCA, embedding.

## Thứ tự chạy
1. Ô 0, 1 ở đây: lập kế hoạch GEE, ghi `HP_LS_CHUNG/ke_hoach_ls.json`. Chia sẻ thư mục `HP_LS_CHUNG` (quyền **Người chỉnh sửa**)
   cho các tài khoản phụ; mỗi tài khoản phụ vào "Được chia sẻ với tôi", chọn **Thêm lối tắt vào Drive**.
2. Mỗi tài khoản phụ mở `ls_GEE_tkX.ipynb`, gửi việc, đợi xong, chạy ô **CHÉP**.
3. Quay lại đây, chạy từ ô 3 trở đi. Năm nào chưa có ảnh thì bước sau tự bỏ qua; chạy lại khi đủ.

Mã truy cập Hugging Face chỉ đọc từ Colab Secrets (`HF_TOKEN`) hoặc hỏi bằng ô nhập ẩn, không ghi vào notebook.

In [ ]:
# ======================= 0. THAM SỐ =======================
HF_REPO      = "lopmaybay/haiphong-lop-tham-chieu"   # bộ dữ liệu của geoportal
PHUONG_AN    = "B"           # "B": ảnh dựng lại trên GEE (khuyến nghị); "A": ảnh tổng hợp có sẵn trên Drive
NAM          = list(range(1987, 2027))
DANH_SACH_TK = ["tk1", "tk2", "tk3", "tk4"]           # tài khoản GEE phụ, theo thứ tự chia việc
THU_MUC_A    = None          # thư mục chứa HP-YYYY-Mosaic.tif (phương án A); None = tự tìm trong My Drive
MAU_A        = "HP-*-Mosaic.tif"
LAM = dict(kiem_ke=True, chuan_hoa=True, cau_noi=True, cog=True, pca=True, emb=True, manifest=True)
K_PCA        = 5             # số PC đưa lên geoportal (lspc1..5); số PC đề xuất bằng thực nghiệm được in ở bước PCA
EMB_CFG      = dict(chung_s2=False, steps=1500)        # chung_s2=True: căn embedding theo g7 của S2 (2017-2026)
GUI_LEN      = False         # True: đẩy lên Hugging Face và cập nhật manifest (ô cuối)
LAM_LAI      = False         # True: làm lại cả bước đã có kết quả
GHI_DE_KE_HOACH = False      # True: lập lại kế hoạch GEE (phải gửi lại mọi việc GEE)

!pip install -q geopandas rasterio huggingface_hub
from google.colab import drive; drive.mount("/content/drive")
import os, sys, glob, json, re, time, shutil
import numpy as np, pandas as pd, geopandas as gpd, rasterio
D = "/content/drive/MyDrive"; MOD = f"{D}/HP_modules"; DRIVE_MY = f"{D}/HP_multiyear"
LSMOD = "/content/ls_mod"; KQ = f"{D}/HP_LS_KETQUA"; TMP = "/content/tmp_ls"
HF_LS = f"{D}/HP_HF_LANDSAT/{PHUONG_AN}"          # bản sao các tệp sẽ đẩy, đúng đường dẫn trong kho HF (mỗi phương án một thư mục)
for p in (LSMOD, KQ, HF_LS, TMP):
    os.makedirs(p, exist_ok=True)


def phien_ban(p, mau):
    if not os.path.exists(p):
        return (0,)
    m = re.search(mau, open(p, encoding="utf-8", errors="ignore").read())
    return tuple(int(v) for v in m.group(1).split(".")) if m else (0,)


if phien_ban(f"{MOD}/s2_hf_lop.py", r'__version__ = "([\d.]+)"') < (1, 2) or not os.path.exists(f"{MOD}/s2_pixlab.py"):
    print("Tải lên HP_modules: s2_hf_lop.py (bản 1.2 trở lên) và s2_pixlab.py"); from google.colab import files
    for k in files.upload():
        shutil.copy(k, MOD)
for p in (MOD, LSMOD):
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)                    # ls_mod trước HP_modules
print("thư mục kết quả:", KQ, "| bản sao kho HF:", HF_LS)

In [ ]:
%%writefile /content/ls_mod/ls_landsat.py
# =============================================================================
# LANDSAT 1987-2026 CHO GEOPORTAL HẢI PHÒNG: phần Python thuần (không cần Earth Engine, không cần GPU)
#   ① kiểm kê ảnh Landsat có sẵn trên Drive (loại tệp, hệ toạ độ, thang giá trị, độ phủ ranh giới, mùa);
#   ② phương án A: chuẩn hoá ảnh tổng hợp có sẵn (HP-YYYY-Mosaic.tif) về lưới 30 m của geoportal;
#   ③ chuẩn hoá bức xạ tương đối bằng IR-MAD (Canty & Nielsen 2008) về một ảnh tham chiếu chung;
#   ④ cầu nối Landsat - Sentinel-2 trên các năm chung (chẩn đoán, không sửa ảnh);
#   ⑤ COG, ảnh xem nhanh, kéo giãn cố định, mục manifest, đẩy lên Hugging Face.
# Lưới: EPSG:32648, ô 30 m, CÙNG GỐC với lưới 10 m của ảnh S2_HP_{năm}: mỗi điểm ảnh Landsat trùng khít 3 × 3 điểm ảnh S2.
# =============================================================================
import os, re, glob, json, math, time, hashlib
import numpy as np

__version__ = "1.0"
CRS = "EPSG:32648"
GOC_LUOI = (615610.0, 2352020.0)          # gốc lưới của ảnh tổng hợp S2_HP_{năm} (giống pcats/pca_ts_py.py)
RES = 30.0
TEN_BANG = ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
BANG_S2 = ["B2", "B3", "B4", "B8", "B11", "B12"]       # tên băng S2 tương đương: geoportal dùng chung công thức chỉ số
BANG_PHU = ["TEMP", "NOBS", "NGUON", "CAMBIEN"]
BANG_RA = TEN_BANG + BANG_PHU
NODATA = -32768
HE_SO = 10000                              # phản xạ × 10000 (như DN của S2 L2A)
HE_SO_T = 100                              # nhiệt độ bề mặt °C × 100
# cờ cảm biến (băng CAMBIEN, cộng bit)
BIT_CB = {"L4": 1, "L5": 1, "L7": 2, "L8": 4, "L9": 8}
# cờ cửa sổ thời gian (băng NGUON): 0 mùa khô đúng năm, 1 mùa khô nới (10 → 5), 2 một ảnh trong mùa,
# 3 mượn mùa khô năm trước và năm sau, 9 ảnh có sẵn không rõ cửa sổ (phương án A)
NGUON_TEN = {0: "mùa khô đúng năm (≥ 2 ảnh)", 1: "mùa khô nới tháng 10 đến 5 (≥ 2 ảnh)", 2: "chỉ một ảnh trong mùa",
             3: "mượn mùa khô năm trước và sau", 9: "ảnh có sẵn, không rõ cửa sổ"}
ALIAS = {"BLUE": ["BLUE", "B", "SR_B1_TM", "B1_TM"], "GREEN": ["GREEN", "G"], "RED": ["RED", "R"], "NIR": ["NIR", "N"],
         "SWIR1": ["SWIR1", "SWIR_1", "S1"], "SWIR2": ["SWIR2", "SWIR_2", "S2"], "TEMP": ["TEMP", "ST", "LST", "TEMPERATURE"],
         "NOBS": ["CLEAR_COUNT", "NOBS", "COUNT", "N_OBS", "CLEARCOUNT"]}


# --------------------------------------------------------------------------- ① lưới
def khung_luoi(bounds, res=RES, goc=GOC_LUOI):
    """Khung chữ nhật phủ `bounds` (EPSG:32648) và nằm đúng mép điểm ảnh của lưới gốc."""
    x0, y0 = goc
    xmin, ymin, xmax, ymax = bounds
    c0, c1 = math.floor((xmin - x0) / res), math.ceil((xmax - x0) / res)
    r0, r1 = math.floor((y0 - ymax) / res), math.ceil((y0 - ymin) / res)
    return dict(ct=[float(res), 0.0, x0, 0.0, -float(res), y0],
                khung=[x0 + c0 * res, y0 - r1 * res, x0 + c1 * res, y0 - r0 * res],
                rong=int(c1 - c0), cao=int(r1 - r0), cot0=int(c0), hang0=int(r0), res=float(res))


def luoi_tu_ranh(ranh, vien_m=500.0, res=RES):
    """ranh: GeoDataFrame / GeoSeries / hình shapely (EPSG:4326 hoặc có crs) -> lưới 30 m phủ ranh giới nới vien_m."""
    import geopandas as gpd
    if hasattr(ranh, "to_crs"):
        g = ranh.to_crs(CRS)
        hinh = g.union_all() if hasattr(g, "union_all") else g.unary_union
    else:
        hinh = gpd.GeoSeries([ranh], crs=4326).to_crs(CRS).iloc[0]
    return khung_luoi(hinh.buffer(vien_m).bounds, res)


def transform_luoi(L):
    from rasterio.transform import from_origin
    return from_origin(L["khung"][0], L["khung"][3], L["res"], L["res"])


# --------------------------------------------------------------------------- ② kiểm kê
_CANH = re.compile(r"(L[TEC]0[4-9])_(L1TP|L1GT|L1GS|L2SP|L2SR)_(\d{3})(\d{3})_(\d{8})_(\d{8})_(\d{2})_(T1|T2|RT)", re.I)


def nhan_dang_ten(path):
    """Nhận dạng tệp theo tên: cảnh USGS (cảm biến, mức xử lý, hàng-cột, ngày, collection, tier) hoặc ảnh tổng hợp năm."""
    ten = os.path.basename(path)
    m = _CANH.search(ten)
    if m:
        vt = {"LT04": "L4", "LT05": "L5", "LE07": "L7", "LC08": "L8", "LC09": "L9"}.get(m.group(1).upper(), m.group(1))
        return dict(kieu="canh", cam_bien=vt, muc=m.group(2).upper(), path=int(m.group(3)), row=int(m.group(4)),
                    ngay=m.group(5), collection=int(m.group(7)), tier=m.group(8).upper(), nam=int(m.group(5)[:4]))
    ys = [int(x) for x in re.findall(r"(?<!\d)((?:19[89]\d|20[0-3]\d))(?!\d)", ten)]
    return dict(kieu="tong_hop_nam" if ys else "khac", nam=ys[0] if ys else None)


def chi_so_bang(mo_ta, so_bang):
    """{tên chuẩn: chỉ số 1-based} từ mô tả băng; không có mô tả thì theo thứ tự B,G,R,NIR,SWIR1,SWIR2,[TEMP],[NOBS]."""
    d = [str(x or "").strip().upper() for x in (mo_ta or [])]
    out = {}
    for k, al in ALIAS.items():
        for i, x in enumerate(d):
            if x in al:
                out[k] = i + 1; break
    if not all(k in out for k in TEN_BANG):
        if so_bang >= 6:
            out = {k: i + 1 for i, k in enumerate(TEN_BANG)}
            if so_bang >= 7: out["TEMP"] = 7
            if so_bang >= 8: out["NOBS"] = 8
    return out


def kieu_gia_tri(mau_nir, dtype):
    """Thang giá trị của băng phản xạ (lấy từ NIR): 'phan_xa' (0..1), 'dn_c2' (DN Collection 2, 7273..43636),
    'x10000' (phản xạ × 10000) hoặc 'khong_ro'."""
    v = np.asarray(mau_nir, float); v = v[np.isfinite(v)]
    if not len(v):
        return "khong_ro"
    p50, p98 = np.percentile(v, 50), np.percentile(v, 98)
    if str(dtype).startswith("float") and p98 < 2.0:
        return "phan_xa"
    if 7000 <= p50 <= 25000 and p98 < 45000:
        return "dn_c2"
    if 300 <= p98 <= 12000:
        return "x10000"
    return "khong_ro"


def ve_phan_xa(a, kieu):
    """Mảng giá trị gốc -> phản xạ (float)."""
    a = np.asarray(a, np.float32)
    if kieu == "phan_xa":
        return a
    if kieu == "dn_c2":
        return a * 0.0000275 - 0.2
    if kieu == "x10000":
        return a / 10000.0
    raise ValueError(f"không rõ thang giá trị ({kieu})")


def _doc_thu_nho(ds, bands, max_canh=600):
    k = max(1, int(math.ceil(max(ds.width, ds.height) / max_canh)))
    a = ds.read(bands, out_shape=(len(bands), max(1, ds.height // k), max(1, ds.width // k)), masked=True).astype(np.float64)
    return a.filled(np.nan), k


def kiem_ke(thu_muc, ranh=None, mau="**/*.tif", bao_cao_gee=None, in_ra=True):
    """
    Kiểm kê mọi GeoTIFF Landsat trong thư mục (đệ quy). Mỗi dòng: tệp, loại, năm, hệ toạ độ, cỡ điểm ảnh (m),
    số băng và tên băng, kiểu dữ liệu, thang giá trị, phân vị phản xạ, nhiệt độ trung vị (dấu hiệu mùa), số lần quan
    sát, độ phủ ranh giới (khung và dữ liệu hợp lệ), cảnh báo. `ranh`: GeoDataFrame ranh giới Hải Phòng mới.
    `bao_cao_gee`: CSV báo cáo cảnh của GEE (như HP_Mosaic_Images_Report_*.csv) để đếm cảnh mùa khô theo năm.
    """
    import pandas as pd
    import rasterio
    from rasterio.warp import transform_geom
    from shapely.geometry import shape, box, mapping
    tep = sorted(set(glob.glob(os.path.join(thu_muc, mau), recursive=True)))
    tep = [t for t in tep if not t.lower().endswith((".aux.xml", ".ovr"))]
    hinh = None
    if ranh is not None:
        g = ranh.to_crs(4326) if hasattr(ranh, "to_crs") else ranh
        hinh = g.union_all() if hasattr(g, "union_all") else (g.unary_union if hasattr(g, "unary_union") else g)
    dong = []
    for t in tep:
        r = dict(tep=os.path.relpath(t, thu_muc), **nhan_dang_ten(t))
        try:
            with rasterio.open(t) as ds:
                r.update(crs=str(ds.crs), w=ds.width, h=ds.height, so_bang=ds.count, kieu_du_lieu=ds.dtypes[0],
                         nodata=ds.nodata, mo_ta=",".join(str(x) for x in (ds.descriptions or []) if x))
                if ds.crs and ds.crs.is_geographic:
                    lat = (ds.bounds.top + ds.bounds.bottom) / 2
                    r["res_m"] = round(abs(ds.transform.a) * 111320 * math.cos(math.radians(lat)), 1)
                else:
                    r["res_m"] = round(abs(ds.transform.a), 2)
                idx = chi_so_bang(ds.descriptions, ds.count)
                if all(k in idx for k in TEN_BANG):
                    a, k = _doc_thu_nho(ds, [idx[b] for b in TEN_BANG])
                    r["thang"] = kieu_gia_tri(a[3], ds.dtypes[0])
                    if r["thang"] != "khong_ro":
                        rf = ve_phan_xa(a, r["thang"])
                        hop = np.isfinite(rf).all(0)
                        if ds.nodata is not None:
                            hop &= ~(a == ds.nodata).any(0)
                        r["ti_le_hop_le"] = round(float(hop.mean()), 3)
                        for i, b in enumerate(TEN_BANG):
                            v = rf[i][hop]
                            if len(v):
                                r[f"{b}_p50"] = round(float(np.percentile(v, 50)), 4)
                        r["phan_xa_ngoai_0_1"] = round(float(((rf > 1.0) | (rf < -0.05)).any(0)[hop].mean()), 4) if hop.any() else None
                    if "TEMP" in idx:
                        tt, _ = _doc_thu_nho(ds, [idx["TEMP"]])
                        v = tt[0][np.isfinite(tt[0])]
                        if len(v):
                            p50 = float(np.percentile(v, 50))
                            r["TEMP_p50"] = round(p50 - 273.15 if p50 > 150 else p50, 2)
                    if "NOBS" in idx:
                        nn, _ = _doc_thu_nho(ds, [idx["NOBS"]])
                        v = nn[0][np.isfinite(nn[0])]
                        if len(v):
                            r["NOBS_p50"] = float(np.percentile(v, 50)); r["NOBS_min"] = float(v.min())
                if hinh is not None and ds.crs:
                    hb = shape(transform_geom("EPSG:4326", ds.crs, mapping(hinh)))
                    k_ = box(*ds.bounds)
                    r["phu_khung_pct"] = round(100 * hb.intersection(k_).area / max(hb.area, 1e-12), 1)
                    # độ phủ dữ liệu hợp lệ: 3000 điểm ngẫu nhiên trong ranh giới
                    rng = np.random.default_rng(0)
                    x0, y0, x1, y1 = hb.bounds
                    xs, ys = rng.uniform(x0, x1, 20000), rng.uniform(y0, y1, 20000)
                    trong = _trong_hinh(hb, xs, ys)
                    xs, ys = xs[trong][:3000], ys[trong][:3000]
                    if len(xs):
                        rr, cc = rasterio.transform.rowcol(ds.transform, xs, ys)
                        rr, cc = np.asarray(rr), np.asarray(cc)
                        tr = (rr >= 0) & (rr < ds.height) & (cc >= 0) & (cc < ds.width)
                        ok = np.zeros(len(xs), bool)
                        if tr.any() and all(k in idx for k in TEN_BANG):
                            b1 = idx["NIR"]
                            vals = np.array([v[0] for v in ds.sample(list(zip(xs[tr], ys[tr])), indexes=b1)], float)
                            hl = np.isfinite(vals)
                            if ds.nodata is not None:
                                hl &= vals != ds.nodata
                            ok[np.where(tr)[0][hl]] = True
                        r["phu_du_lieu_pct"] = round(100 * ok.mean(), 1)
        except Exception as e:                                   # noqa: BLE001
            r["loi"] = f"{type(e).__name__}: {e}"[:160]
        dong.append(r)
    df = pd.DataFrame(dong)
    if bao_cao_gee and os.path.exists(bao_cao_gee):
        mk = mua_kho_tu_bao_cao(bao_cao_gee)
        df = df.merge(mk, how="left", on="nam") if "nam" in df else df
    df["canh_bao"] = df.apply(_canh_bao, axis=1) if len(df) else []
    if in_ra and len(df):
        cot = [c for c in ["tep", "kieu", "nam", "crs", "res_m", "so_bang", "thang", "TEMP_p50", "NOBS_p50", "phu_khung_pct",
                           "phu_du_lieu_pct", "canh_mua_kho", "canh_bao"] if c in df]
        print(df[cot].to_string(index=False, max_colwidth=70))
    return df


def _trong_hinh(hinh, xs, ys):
    try:
        import shapely
        return shapely.contains_xy(hinh, xs, ys)                 # shapely >= 2.0
    except AttributeError:
        from shapely import vectorized
        return vectorized.contains(hinh, xs, ys)


def mua_kho_tu_bao_cao(csv):
    """Từ CSV báo cáo cảnh của GEE (cột Image_ID kiểu 2_LT05_126045_19870118, Cloud_Cover_Percent):
    số cảnh trong mùa khô năm y (1/11/(y-1) đến 30/4/y), số cảnh mây < 50 %, các cảm biến, các tháng có cảnh trong năm."""
    import pandas as pd
    d = pd.read_csv(csv)
    s = d["Image_ID"].astype(str)
    d["ngay"] = pd.to_datetime(s.str.extract(r"_(\d{8})$")[0], format="%Y%m%d", errors="coerce")
    d["cb"] = s.str.extract(r"(L[TEC]0\d)")[0]
    d = d.dropna(subset=["ngay"])
    out = []
    for y in sorted(set(d["ngay"].dt.year) | set(d["ngay"].dt.year + 1)):
        m = d[(d.ngay >= f"{y - 1}-11-01") & (d.ngay < f"{y}-05-01")]
        cn = d[d.ngay.dt.year == y]
        if not len(cn) and not len(m):
            continue
        out.append(dict(nam=int(y), canh_mua_kho=len(m),
                        canh_mua_kho_may50=int((m["Cloud_Cover_Percent"] < 50).sum()) if "Cloud_Cover_Percent" in m else None,
                        cam_bien_bao_cao=",".join(sorted(set(cn["cb"].dropna()))),
                        thang_co_canh=" ".join(str(x) for x in sorted(set(cn.ngay.dt.month)))))
    return pd.DataFrame(out)


def _canh_bao(r):
    w = []
    def g(k):
        v = r.get(k) if hasattr(r, "get") else None
        return None if v is None or (isinstance(v, float) and math.isnan(v)) else v
    if g("loi"): w.append("không đọc được")
    if str(g("crs") or "").endswith("4326"): w.append("hệ toạ độ địa lý: phải nắn lại (mất một lần lấy mẫu lại)")
    if g("phu_khung_pct") is not None and g("phu_khung_pct") < 95: w.append(f"chỉ phủ {g('phu_khung_pct')} % Hải Phòng mới")
    if g("phan_xa_ngoai_0_1") and g("phan_xa_ngoai_0_1") > 0.001: w.append("có phản xạ ngoài 0..1 (bão hoà, chưa lọc QA_RADSAT?)")
    th = str(g("thang_co_canh") or "")
    if th and len(th.split()) > 7: w.append("ảnh tổng hợp gộp nhiều mùa (tháng " + th + ")")
    if g("canh_mua_kho_may50") is not None and g("canh_mua_kho_may50") < 2: w.append("mùa khô có dưới 2 cảnh ít mây")
    if g("NOBS_min") is not None and g("NOBS_min") < 2: w.append("có điểm ảnh chỉ 1 lần quan sát")
    return "; ".join(w)


def thieu_nam(df, tu=1987, den=2026):
    co = set(int(x) for x in df["nam"].dropna()) if "nam" in df else set()
    return [y for y in range(tu, den + 1) if y not in co]


# --------------------------------------------------------------------------- ③ phương án A: chuẩn hoá ảnh có sẵn
def cb_theo_nam(y):
    """Cảm biến của ảnh có sẵn khi không có thông tin từng điểm ảnh (APP-TONGHOP dùng L5 đến 2011, L8/9 từ 2013)."""
    if y <= 2011: return BIT_CB["L5"]
    if y >= 2021: return BIT_CB["L8"] | BIT_CB["L9"]
    return BIT_CB["L8"]


def chuan_hoa_anh(src, out, L, nam, thang=None, lay_mau="bilinear", nguon=9, cam_bien=None, tmp_dir="/tmp", verbose=True):
    """
    Ảnh tổng hợp có sẵn (mọi hệ toạ độ, phản xạ 0..1 hoặc DN) -> GeoTIFF int16 10 băng trên lưới L (EPSG:32648, 30 m):
    BLUE..SWIR2 (phản xạ × 10000), TEMP (°C × 100), NOBS, NGUON, CAMBIEN; nodata -32768.
    Phản xạ và nhiệt độ lấy mẫu song tuyến (bilinear), NOBS láng giềng gần nhất. Trả về đường dẫn tệp (GTiff thường,
    chưa phải COG: gọi lam_cog sau khi chuẩn hoá tương đối).
    """
    import rasterio
    from rasterio.enums import Resampling
    from rasterio.vrt import WarpedVRT
    from rasterio.windows import Window
    t0 = time.time()
    tf, W, H = transform_luoi(L), L["rong"], L["cao"]
    with rasterio.open(src) as ds:
        idx = chi_so_bang(ds.descriptions, ds.count)
        if not all(k in idx for k in TEN_BANG):
            raise ValueError(f"{src}: không nhận ra đủ 6 băng phản xạ (mô tả {ds.descriptions})")
        if thang is None:
            a, _ = _doc_thu_nho(ds, [idx["NIR"]])
            thang = kieu_gia_tri(a[0], ds.dtypes[0])
        kw = dict(crs=CRS, transform=tf, width=W, height=H)
        if ds.nodata is not None:
            kw.update(src_nodata=ds.nodata, nodata=ds.nodata)
        elif np.dtype(ds.dtypes[0]).kind == "f":
            kw.update(src_nodata=np.nan, nodata=np.nan)     # ảnh GEE float không ghi nodata: NaN ngoài vùng; ngoài khung ảnh gốc cũng NaN thay vì 0
        prof = dict(driver="GTiff", width=W, height=H, count=len(BANG_RA), dtype="int16", crs=CRS, transform=tf,
                    nodata=NODATA, tiled=True, blockxsize=256, blockysize=256, compress="deflate", predictor=2, BIGTIFF="IF_SAFER")
        os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
        cbv = cam_bien if cam_bien is not None else cb_theo_nam(nam)
        with WarpedVRT(ds, resampling=getattr(Resampling, lay_mau), **kw) as v, \
                WarpedVRT(ds, resampling=Resampling.nearest, **kw) as vn, rasterio.open(out, "w", **prof) as o:
            for r0 in range(0, H, 512):
                hh = min(512, H - r0); win = Window(0, r0, W, hh)
                a = v.read([idx[b] for b in TEN_BANG], window=win, masked=True).astype(np.float32)
                m = np.ma.getmaskarray(a).any(0) | ~np.isfinite(a.filled(np.nan)).all(0)
                rf = ve_phan_xa(a.filled(np.nan), thang)
                ra = np.full((len(BANG_RA), hh, W), NODATA, np.int16)
                ra[:6] = np.clip(np.round(np.nan_to_num(rf) * HE_SO), -32767, 32767).astype(np.int16)
                if "TEMP" in idx:
                    t_ = v.read(idx["TEMP"], window=win, masked=True).astype(np.float32).filled(np.nan)
                    t_ = np.where(t_ > 150, t_ - 273.15, t_)               # Kelvin -> °C
                    ra[6] = np.where(np.isfinite(t_), np.clip(np.round(t_ * HE_SO_T), -32767, 32767), NODATA).astype(np.int16)
                if "NOBS" in idx:
                    n_ = vn.read(idx["NOBS"], window=win, masked=True).astype(np.float32).filled(np.nan)
                    ra[7] = np.where(np.isfinite(n_), np.clip(n_, 0, 32767), NODATA).astype(np.int16)
                    m |= ~(n_ > 0)                                           # không có lần quan sát nào: trống
                ra[8] = nguon; ra[9] = cbv
                ra[:, m] = NODATA
                o.write(ra, window=win)
            o.descriptions = tuple(BANG_RA)
            o.update_tags(nam=str(nam), thang_goc=thang, nguon_goc=os.path.basename(src), phuong_an="A")
    if verbose:
        print(f"    {os.path.basename(out)}: {W}×{H}, thang gốc {thang}, {time.time() - t0:.0f}s")
    return out


# --------------------------------------------------------------------------- ④ IR-MAD và chuẩn hoá tương đối
def irmad(X, Y, lap=50, sai_so=1e-5):
    """
    IR-MAD (Nielsen 2007; Canty & Nielsen 2008): tương quan chính tắc giữa X và Y (N × p), biến MAD, trọng số = xác
    suất KHÔNG đổi theo χ²(p) của tổng bình phương biến MAD chuẩn hoá; lặp đến khi hệ số tương quan chính tắc ổn định.
    Trả về dict(p_khong_doi (N,), rho (p,), n_lap).
    """
    from scipy.stats import chi2
    X, Y = np.asarray(X, np.float64), np.asarray(Y, np.float64)
    N, p = X.shape
    w = np.ones(N); rho_cu = None
    for it in range(1, lap + 1):
        sw = w.sum()
        mx, my = w @ X / sw, w @ Y / sw
        Xc, Yc = X - mx, Y - my
        S11 = (Xc * w[:, None]).T @ Xc / sw + 1e-12 * np.eye(p)
        S22 = (Yc * w[:, None]).T @ Yc / sw + 1e-12 * np.eye(p)
        S12 = (Xc * w[:, None]).T @ Yc / sw
        L1 = np.linalg.cholesky(S11)
        Li = np.linalg.inv(L1)
        C = Li @ S12 @ np.linalg.solve(S22, S12.T) @ Li.T
        ev, U = np.linalg.eigh((C + C.T) / 2)
        o = np.argsort(ev)[::-1]; ev, U = ev[o], U[:, o]
        rho = np.sqrt(np.clip(ev, 0, 1))
        A = Li.T @ U                                             # a' S11 a = 1
        B = np.linalg.solve(S22, S12.T) @ A / np.maximum(rho, 1e-12)
        B /= np.sqrt(np.maximum(np.einsum("ij,jk,ki->i", B.T, S22, B), 1e-24))[None, :]   # b' S22 b = 1
        sg = np.sign(np.einsum("ij,jk,ki->i", A.T, S12, B)); B *= np.where(sg == 0, 1, sg)[None, :]
        M = Xc @ A - Yc @ B
        var = np.maximum(2 * (1 - rho), 1e-12)
        Z = (M * M / var).sum(1)
        w = 1 - chi2.cdf(Z, p)
        if rho_cu is not None and np.max(np.abs(rho - rho_cu)) < sai_so:
            break
        rho_cu = rho
    return dict(p_khong_doi=w, rho=rho, n_lap=it, mad=M)


def hoi_quy_truc_giao(x, y):
    """Hồi quy trực giao (tổng bình phương khoảng cách vuông góc): y ≈ a x + b. Trả về a, b, r²."""
    x, y = np.asarray(x, np.float64), np.asarray(y, np.float64)
    mx, my = x.mean(), y.mean()
    sxx, syy, sxy = ((x - mx) ** 2).mean(), ((y - my) ** 2).mean(), ((x - mx) * (y - my)).mean()
    a = (syy - sxx + math.sqrt((syy - sxx) ** 2 + 4 * sxy * sxy)) / (2 * sxy) if abs(sxy) > 1e-18 else 1.0
    r2 = sxy * sxy / max(sxx * syy, 1e-24)
    return float(a), float(my - a * mx), float(r2)


# p_min: xác suất KHÔNG đổi tối thiểu để là điểm ảnh bất biến (Canty & Nielsen dùng 0.95).
# Xác suất tính lại bằng thang ĐỘ LỆCH TUYỆT ĐỐI TRUNG VỊ của từng biến MAD ở vòng cuối: trọng số IR-MAD thu hẹp dần
# phương sai ước lượng, nên khi hai ảnh gần giống nhau (năm OLI so với tham chiếu OLI) gần như mọi điểm ảnh bị coi là
# "đổi". Thang trung vị bền với cả hai chiều (đuôi dày do đổi thật, phương sai ước lượng sụp).
# Khi ít hơn n_pif_muc điểm vượt p_min: lấy n_pif_muc điểm có xác suất cao nhất trong số điểm KHÔNG có ý nghĩa thay đổi
# ở mức 5 % (p > 0.05). n_pif_min: dưới mức này không chuẩn hoá năm đó.
# Từng băng: hồi quy trực giao nếu r² ≥ r2_min và hệ số góc trong a_khoang; không thì hệ số TỈ LỆ trung bình trên điểm bất
# biến (chỉ sửa độ lợi, chặn 0) nếu r² ≥ r2_min_ti_le và tỉ lệ trong a_khoang; không nữa thì giữ nguyên băng đó.
# Ghi rõ cách cho từng băng.
CONG_MAC = dict(p_min=0.95, n_pif_muc=5000, n_pif_min=500, r2_min=0.85, r2_min_ti_le=0.5, a_khoang=(0.7, 1.43),
                n_mau=300000)


def xac_suat_ben(M):
    """Xác suất KHÔNG đổi theo χ²(p) với biến MAD chuẩn hoá bằng trung vị và 1.4826 × độ lệch tuyệt đối trung vị."""
    from scipy.stats import chi2
    M = np.asarray(M, np.float64)
    med = np.median(M, 0)
    s = 1.4826 * np.median(np.abs(M - med), 0)
    s = np.where(s > 1e-12, s, M.std(0) + 1e-12)
    return 1 - chi2.cdf((((M - med) / s) ** 2).sum(1), M.shape[1])


def doc_cap(tep, tep_tc, bang=6, buoc=2, n_mau=300000, seed=0):
    """Giá trị phản xạ (×10000) của các điểm ảnh hợp lệ ở CẢ hai ảnh (cùng lưới), lấy thưa `buoc` rồi chọn ngẫu nhiên."""
    import rasterio
    from rasterio.enums import Resampling
    with rasterio.open(tep) as a, rasterio.open(tep_tc) as b:
        if (a.width, a.height, tuple(a.transform)[:6]) != (b.width, b.height, tuple(b.transform)[:6]):
            raise ValueError("hai ảnh không cùng lưới")
        sh = (bang, max(1, a.height // buoc), max(1, a.width // buoc))
        X = a.read(list(range(1, bang + 1)), out_shape=sh, resampling=Resampling.nearest).reshape(bang, -1).T.astype(np.float64)
        Y = b.read(list(range(1, bang + 1)), out_shape=sh, resampling=Resampling.nearest).reshape(bang, -1).T.astype(np.float64)
    ok = ((X != NODATA).all(1) & (Y != NODATA).all(1) & (X > -2000).all(1) & (Y > -2000).all(1)
          & (X < 15000).all(1) & (Y < 15000).all(1))
    X, Y = X[ok], Y[ok]
    if len(X) > n_mau:
        k = np.random.default_rng(seed).choice(len(X), n_mau, replace=False); X, Y = X[k], Y[k]
    return X, Y


def he_so_chuan_hoa(X, Y, cong=None):
    """Từ cặp (X: ảnh cần chuẩn hoá, Y: tham chiếu): IR-MAD chọn điểm ảnh bất biến, hệ số từng băng (xem CONG_MAC).
    Trả về dict(a, b, r2, cach, n_pif, dat, ly_do) với Y ≈ a X + b; `dat` = năm qua cổng (đủ điểm bất biến, ít nhất
    một băng sửa được)."""
    c = dict(CONG_MAC, **(cong or {}))
    nb = X.shape[1]
    if len(X) < c["n_pif_min"]:
        return dict(a=[1.0] * nb, b=[0.0] * nb, r2=[0.0] * nb, cach=["giu"] * nb, n_pif=0, dat=False,
                    ly_do="quá ít điểm ảnh chung")
    R = irmad(X, Y)
    p = xac_suat_ben(R["mad"])
    pif = p > c["p_min"]
    if pif.sum() < c["n_pif_muc"]:
        ung = np.where(p > 0.05)[0]
        k = min(len(ung), c["n_pif_muc"])
        pif = np.zeros(len(p), bool); pif[ung[np.argsort(p[ung])[::-1][:k]]] = True
    n = int(pif.sum())
    a, b, r2, cach = [], [], [], []
    lo, hi = c["a_khoang"]
    for j in range(nb):
        aj, bj, rj = hoi_quy_truc_giao(X[pif, j], Y[pif, j]) if n > 10 else (1.0, 0.0, 0.0)
        tl = float(Y[pif, j].mean() / X[pif, j].mean()) if n > 10 and abs(X[pif, j].mean()) > 1e-9 else 1.0
        if n >= c["n_pif_min"] and rj >= c["r2_min"] and lo <= aj <= hi:
            cj = "truc_giao"
        elif n >= c["n_pif_min"] and rj >= c["r2_min_ti_le"] and lo <= tl <= hi:
            aj, bj, cj = tl, 0.0, "ti_le"
        else:
            aj, bj, cj = 1.0, 0.0, "giu"
        a.append(round(aj, 5)); b.append(round(bj, 2)); r2.append(round(rj, 4)); cach.append(cj)
    ly = []
    if n < c["n_pif_min"]:
        ly.append(f"chỉ {n} điểm ảnh bất biến")
    if any(x != "truc_giao" for x in cach):
        ly.append("băng " + ", ".join(f"{TEN_BANG[j] if j < len(TEN_BANG) else j}: {x}" for j, x in enumerate(cach)
                                      if x != "truc_giao"))
    dat = n >= c["n_pif_min"] and any(x != "giu" for x in cach)
    return dict(a=a, b=b, r2=r2, cach=cach, n_pif=n, ti_le_pif=round(n / len(X), 4),
                rho=[round(float(x), 4) for x in R["rho"]], n_lap=R["n_lap"], dat=dat, ly_do="; ".join(ly))


def ap_chuan_hoa(src, out, hs, bang=6):
    """Áp Y = a X + b cho `bang` băng phản xạ đầu (giữ nodata, giữ các băng còn lại), ghi tệp mới."""
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(src) as ds:
        prof = ds.profile.copy()
        with rasterio.open(out, "w", **prof) as o:
            for r0 in range(0, ds.height, 512):
                hh = min(512, ds.height - r0); win = Window(0, r0, ds.width, hh)
                a = ds.read(window=win)
                for j in range(bang):
                    v = a[j]; ok = v != NODATA
                    v2 = np.round(v.astype(np.float64) * hs["a"][j] + hs["b"][j])
                    a[j] = np.where(ok, np.clip(v2, -32767, 32767), NODATA).astype(np.int16)
                o.write(a, window=win)
            o.descriptions = ds.descriptions
            tags = ds.tags(); tags.update(chuan_hoa=json.dumps({k: hs[k] for k in ("a", "b", "r2", "n_pif")}))
            o.update_tags(**tags)
    return out


# --------------------------------------------------------------------------- ⑤ cầu nối Landsat - Sentinel-2 (chẩn đoán)
S2_THU_TU = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]      # thứ tự băng của S2_HP_{năm}


def cau_noi_s2(ls_tif, s2_tif, so_khoi=40, canh=200, seed=0):
    """So sánh phản xạ Landsat (30 m) với S2 (10 m, gộp trung bình 3 × 3 trên lưới chung) cùng năm, từng băng.
    Trả về dict băng -> n, OLS (a, b), r², độ lệch trung bình và RMSE (đơn vị phản xạ). Chỉ chẩn đoán, không sửa ảnh."""
    import rasterio
    from rasterio.windows import Window
    rng = np.random.default_rng(seed)
    acc = {b: ([], []) for b in BANG_S2}
    with rasterio.open(ls_tif) as L, rasterio.open(s2_tif) as S:
        if abs(S.transform.a - L.transform.a / 3) > 1e-6:
            raise ValueError("ảnh S2 phải là lưới 10 m cùng gốc (S2_HP_{năm})")
        dx = (L.transform.c - S.transform.c) / S.transform.a
        dy = (S.transform.f - L.transform.f) / S.transform.a
        if abs(dx - round(dx)) > 1e-6 or abs(dy - round(dy)) > 1e-6:
            raise ValueError("hai lưới lệch nhau, không trùng khít 3 × 3")
        dx, dy = int(round(dx)), int(round(dy))
        for _ in range(so_khoi * 5):
            if sum(len(v[0]) for v in acc.values()) >= so_khoi * canh * canh * len(BANG_S2) // 4:
                break
            c0, r0 = int(rng.integers(0, max(1, L.width - canh))), int(rng.integers(0, max(1, L.height - canh)))
            sc, sr = dx + 3 * c0, dy + 3 * r0
            if sc < 0 or sr < 0 or sc + 3 * canh > S.width or sr + 3 * canh > S.height:
                continue
            a = L.read(list(range(1, 7)), window=Window(c0, r0, canh, canh)).astype(np.float64)
            s = S.read([S2_THU_TU.index(b) + 1 for b in BANG_S2], window=Window(sc, sr, 3 * canh, 3 * canh)).astype(np.float64)
            hl = (s > 0).all(0)
            s = np.where(hl, s, np.nan).reshape(6, canh, 3, canh, 3)
            with np.errstate(all="ignore"):
                import warnings
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    sm = np.nanmean(s, axis=(2, 4))
            ok = (a != NODATA).all(0) & np.isfinite(sm).all(0) & (hl.reshape(canh, 3, canh, 3).all(axis=(1, 3)))
            if ok.mean() < 0.3:
                continue
            for j, b in enumerate(BANG_S2):
                acc[b][0].append(a[j][ok] / HE_SO); acc[b][1].append(sm[j][ok] / HE_SO)
    out = {}
    for b, (xs, ys) in acc.items():
        if not xs:
            continue
        x, y = np.concatenate(xs), np.concatenate(ys)
        A = np.vstack([x, np.ones_like(x)]).T
        (aa, bb), *_ = np.linalg.lstsq(A, y, rcond=None)
        r = np.corrcoef(x, y)[0, 1]
        out[b] = dict(n=int(len(x)), a=round(float(aa), 4), b=round(float(bb), 4), r2=round(float(r * r), 4),
                      lech=round(float((x - y).mean()), 4), rmse=round(float(np.sqrt(((x - y) ** 2).mean())), 4))
    return out


# --------------------------------------------------------------------------- ⑥ COG, kéo giãn, xem nhanh
def lam_cog(src, out, blocksize=256):
    """GeoTIFF int16 10 băng -> COG (DEFLATE, predictor 2, overview trung bình) để trình duyệt đọc theo đoạn."""
    import rasterio.shutil as rsh
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    rsh.copy(src, out, driver="COG", compress="DEFLATE", predictor=2, blocksize=blocksize, overviews="AUTO",
             resampling="AVERAGE", BIGTIFF="IF_SAFER", NUM_THREADS="ALL_CPUS")
    return out


def keo_gian(ds_tep, n_win=40, win=200, seed=0, windows=(5, 15), p=(2, 98)):
    """Kéo giãn CỐ ĐỊNH chung mọi năm (như s2d): lo, hi = p2, p98 từng băng phản xạ (gộp mẫu các năm), và p98 độ lệch
    chuẩn CTX ở cửa sổ 5 × 5, 15 × 15 (s5, s15). Đơn vị DN (phản xạ × 10000)."""
    import rasterio
    from rasterio.windows import Window
    from scipy.ndimage import uniform_filter
    rng = np.random.default_rng(seed)
    vals, sds = [], {w: [] for w in windows}
    for t in ds_tep:
        with rasterio.open(t) as ds:
            W, H = ds.width, ds.height
            ww, hh = min(win, W), min(win, H)
            got, tries = 0, 0
            while got < max(1, n_win // max(1, len(ds_tep))) and tries < 400:
                tries += 1
                c0, r0 = int(rng.integers(0, max(1, W - ww + 1))), int(rng.integers(0, max(1, H - hh + 1)))
                a = ds.read(list(range(1, 7)), window=Window(c0, r0, ww, hh)).astype(np.float64)
                ok = (a != NODATA).all(0)
                if ok.mean() < 0.5:
                    continue
                got += 1
                vals.append(a[:, ok])
                for w_ in windows:
                    r = (w_ - 1) // 2
                    inner = ok.copy(); inner[:r, :] = inner[-r:, :] = inner[:, :r] = inner[:, -r:] = False
                    s = []
                    for b in range(6):
                        x = np.where(ok, a[b], 0)
                        m = uniform_filter(x, w_, mode="nearest"); m2 = uniform_filter(x * x, w_, mode="nearest")
                        s.append(np.sqrt(np.maximum(m2 - m * m, 0))[inner])
                    sds[w_].append(np.stack(s))
    V = np.concatenate(vals, 1)
    out = dict(lo=[int(round(x)) for x in np.percentile(V, p[0], 1)], hi=[int(round(x)) for x in np.percentile(V, p[1], 1)])
    for w_ in windows:
        S = np.concatenate(sds[w_], 1)
        out[f"s{w_}"] = [int(round(x)) for x in np.percentile(S, p[1], 1)]
    return out


def anh_xem_nhanh(src, out, grid3857, kg, bang=("RED", "GREEN", "BLUE"), gamma=1.0, tmp_dir="/tmp", verbose=True):
    """Ảnh màu 8 bit trên lưới 3857 (như s2tc): kéo giãn cố định kg (lo, hi) giống nhau mọi năm."""
    import s2_hf_lop as H
    ix = [TEN_BANG.index(b) for b in bang]
    return H.rgb_cog(src, [i + 1 for i in ix], [kg["lo"][i] for i in ix], [kg["hi"][i] for i in ix], out, grid3857,
                     gamma=gamma, nodata=NODATA, tmp_dir=tmp_dir, verbose=verbose)


# --------------------------------------------------------------------------- ⑦ manifest, Hugging Face
def muc_manifest(nam, kg, L, phuong_an, chuan_hoa=None, cau_noi=None, ghi_chu=""):
    """Mục 'ls' của manifest.json (geoportal 3.0 đọc mục này như một nguồn ảnh quang học thứ hai)."""
    return dict(duong_dan="ls/ls_{y}.tif", nam=sorted(int(y) for y in nam), bang=BANG_S2, ten_bang=TEN_BANG, bang_phu=BANG_PHU,
                he_so=HE_SO, he_so_nhiet=HE_SO_T, crs=CRS, res=RES, goc_luoi=list(GOC_LUOI), khung=L["khung"], keo_gian=kg,
                phuong_an=phuong_an, mua="mùa khô: 1/11 năm trước đến 30/4", nguon_ten=NGUON_TEN, bit_cam_bien=BIT_CB,
                nguon="USGS Landsat Collection 2 Level-2 (TM, ETM+, OLI, OLI-2), Tier 1; xử lý: Phạm Đăng Hiển",
                chuan_hoa=chuan_hoa or {}, cau_noi_s2=cau_noi or {}, ghi_chu=ghi_chu, phien_ban_ma=__version__)


def lop_manifest(nam, co_pca=0, pca_kg=None, co_emb=False):
    """Các lớp xem của Landsat (thêm vào manifest['layers']): màu thật, (tuỳ) PC xám, (tuỳ) embedding RGB.
    Trường 'nguon': 'ls' để geoportal tách khỏi lớp Sentinel-2."""
    nam = sorted(int(y) for y in nam)
    ds = [dict(id="lstc", ten="Landsat màu thật (tổng hợp mùa khô)", ten_en="Landsat natural colour (dry-season composite)",
               ten_ru="Landsat в естественных цветах (композит сухого сезона)", kieu="rgb", nguon="ls",
               duong_dan="lstc/lstc_{y}.tif", nam=nam)]
    for q in range(1, co_pca + 1):
        ds.append(dict(id=f"lspc{q}", ten=f"Landsat PC{q} chuỗi năm (đỏ +, xanh −, trắng 0)",
                       ten_en=f"Landsat PC{q} of the annual series (red +, blue −, white 0)",
                       ten_ru=f"Landsat PC{q} годового ряда (красный +, синий −, белый 0)", kieu="xam", nguon="ls",
                       bang_mau_lien_tuc="rdbu", duong_dan=f"lspc{q}/lspc{q}_{{y}}.tif", nam=nam,
                       keo_gian=(pca_kg or {}).get(q)))
    if co_emb:
        for tp, sfx in (("1-2-3", ""), ("4-5-6", "b")):
            ds.append(dict(id=f"lsg{sfx}", ten=f"Embedding Landsat, thành phần {tp}", ten_en=f"Landsat embedding, components {tp}",
                           ten_ru=f"Эмбеддинг Landsat, компоненты {tp}", kieu="rgb", nguon="ls",
                           duong_dan=f"lsg{sfx}/lsg{sfx}_{{y}}.tif", nam=nam, phep_chieu="emb/lsg_phep_chieu.json"))
    return ds


def gop_manifest(man, muc_ls=None, lop=None, muc_lspc=None):
    """Gộp vào manifest hiện có: thay mục cùng id (không trùng lặp), giữ nguyên mọi thứ khác."""
    man = json.loads(json.dumps(man))
    if muc_ls is not None:
        man["ls"] = muc_ls
    if muc_lspc is not None:
        man["lspc"] = muc_lspc
    if lop:
        ids = {l["id"] for l in lop}
        man["layers"] = [l for l in man.get("layers", []) if l.get("id") not in ids] + lop
    man["cap_nhat"] = time.strftime("%Y-%m-%d %H:%M")
    return man


def day_tep(api, repo, local, path_in_repo, lan=4, msg=None):
    """Đẩy một tệp lên kho dữ liệu HF, thử lại khi lỗi mạng (đợi tăng dần)."""
    for k in range(lan):
        try:
            api.upload_file(path_or_fileobj=local, path_in_repo=path_in_repo, repo_id=repo, repo_type="dataset",
                            commit_message=msg or f"Landsat: {path_in_repo}")
            return True
        except Exception as e:                                   # noqa: BLE001
            if k == lan - 1:
                raise
            print(f"    lỗi đẩy {path_in_repo} ({type(e).__name__}), thử lại sau {10 * (k + 1)} s")
            time.sleep(10 * (k + 1))


def da_co_tren_hf(api, repo):
    """Tập đường dẫn đã có trong kho dữ liệu (để chạy lại thì bỏ qua)."""
    try:
        return set(api.list_repo_files(repo, repo_type="dataset"))
    except Exception:                                            # noqa: BLE001
        return set()


def cap_nhat_manifest_hf(api, repo, ham_sua, tmp_dir="/tmp"):
    """Tải manifest.json mới nhất, lưu bản sao an toàn manifest_luu/manifest_{giờ}.json, sửa bằng ham_sua(man) -> man,
    rồi đẩy lại. Không bao giờ ghi đè mà không có bản sao."""
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(repo, "manifest.json", repo_type="dataset", force_download=True, local_dir=tmp_dir)
    man = json.load(open(p, encoding="utf-8"))
    st = time.strftime("%Y%m%d_%H%M")
    day_tep(api, repo, p, f"manifest_luu/manifest_{st}.json", msg="bản sao manifest trước khi thêm Landsat")
    moi = ham_sua(man)
    q = os.path.join(tmp_dir, "manifest_moi.json")
    json.dump(moi, open(q, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    day_tep(api, repo, q, "manifest.json", msg="manifest: thêm Landsat")
    return moi


def ma_bam(d):
    return hashlib.md5(json.dumps(d, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:6]


# --------------------------------------------------------------------------- ⑧ kế hoạch phương án B (GEE) và gom kết quả
# Cấu hình dựng ảnh trên GEE (ls_gee.py dùng). Đặt ở đây để tài khoản chính, không chạy GEE, cũng đọc được mã cấu hình.
CAU_HINH_GEE = dict(
    phien="B1",
    mua=[[11, -1], [4, 0]],             # mùa khô năm y: tháng 11 năm y-1 đến hết tháng 4 năm y (như ảnh S2 của geoportal)
    mua_noi=[[10, -1], [5, 0]],         # nới khi thiếu ảnh: tháng 10 năm y-1 đến hết tháng 5 năm y
    may_canh_max=80,                    # bỏ cảnh mây > 80 % trước khi lọc từng điểm ảnh (bớt việc cho GEE)
    sr_khoang=[-0.05, 1.0],
    opacity_max=0.3,                    # TM, ETM+: SR_ATMOS_OPACITY × 0.001 > 0.3 là khói mù hoặc mây (USGS)
    bo_aerosol_cao=True,                # OLI: SR_QA_AEROSOL bit 6-7 = 11 (mức cao) không nên dùng (USGS)
    bo_mep_px={"L4": 2, "L5": 2, "L7": 2, "L8": 0, "L9": 0},
    l7_den=2016,                        # quỹ đạo Landsat 7 trôi khỏi giờ chụp danh định từ 2017 (USGS)
    l7_du_phong_tu=2014,                # từ 2014 Landsat 7 chỉ dùng khi OLI thiếu ảnh
    n_toi_thieu=2,
    nam_tham_chieu=[2015, 2020])


def ma_cau_hinh_gee(cfg=None):
    return hashlib.md5(json.dumps(cfg or CAU_HINH_GEE, sort_keys=True).encode()).hexdigest()[:6]


def ten_tep_gee(y, cfg=None):
    return f"LS_HP_{int(y)}_{ma_cau_hinh_gee(cfg)}"


def chia_viec(ds_nam, tai_khoan):
    """Năm có mẫu, năm gần đây trước; xen kẽ tài khoản. Trả về {tài khoản: [năm]}."""
    uu = [2025, 2023, 2017, 2011, 1987, 2022, 2020, 2000, 1995, 1990, 2005, 2012, 2013]
    thu_tu = [y for y in uu if y in ds_nam] + [y for y in sorted(ds_nam, reverse=True) if y not in uu]
    out = {tk: [] for tk in tai_khoan}
    for i, y in enumerate(thu_tu):
        out[tai_khoan[i % len(tai_khoan)]].append(y)
    return out


def lap_ke_hoach_gee(bnd, tai_khoan, nam=range(1987, 2027), vien_m=500, sai_so_m=100, cfg=None):
    """Kế hoạch dùng chung cho mọi tài khoản GEE (tài khoản chính ghi ke_hoach_ls.json vào thư mục chung):
    lưới 30 m (khung tính trên ranh giới nới vien_m, CHƯA rút gọn, như luoi_tu_ranh), vùng cắt đã rút gọn (EPSG:4326),
    cấu hình, mã cấu hình và việc của từng tài khoản."""
    import geopandas as gpd
    from shapely.geometry import mapping
    g = bnd.to_crs(CRS)
    hp = g.union_all() if hasattr(g, "union_all") else g.unary_union
    L = khung_luoi(hp.buffer(vien_m).bounds)
    vung = hp.buffer(vien_m).simplify(sai_so_m, preserve_topology=True)
    g4 = gpd.GeoSeries([vung], crs=CRS).to_crs(4326).iloc[0]
    cfg = json.loads(json.dumps(cfg or CAU_HINH_GEE))
    nam = [int(y) for y in nam]
    return dict(phien_ban=__version__, tao_luc=time.strftime("%Y-%m-%d %H:%M"), crs=CRS, L=L,
                vung_geojson=json.loads(json.dumps(mapping(g4))), bao=list(g4.bounds), cfg=cfg,
                ma_cau_hinh=ma_cau_hinh_gee(cfg), nam=nam, tai_khoan=list(tai_khoan), chia=chia_viec(nam, list(tai_khoan)),
                thu_muc_xuat="HP_LS_XUAT_{tk}", thu_muc_chung="HP_LS_CHUNG")


_MANH = re.compile(r"^(LS_HP_(?:TC_\d{4}_\d{4}|\d{4})_[0-9a-f]{6})(-\d{10}-\d{10})?\.tif$")


def tim_xuat_gee(ds_thu_muc, ma=None):
    """{tên gốc: [mảnh]} của mọi tệp xuất GEE (GEE chia ảnh lớn thành mảnh tên đuôi -hàng-cột). Lọc theo mã cấu hình."""
    out = {}
    for tm in ds_thu_muc:
        for p in glob.glob(os.path.join(tm, "**", "LS_HP_*.tif"), recursive=True):
            m = _MANH.match(os.path.basename(p))
            if not m or (ma and not m.group(1).endswith("_" + ma)):
                continue
            out.setdefault(m.group(1), {})[os.path.basename(p)] = p        # cùng tên ở hai thư mục: giữ một
    return {k: sorted(v.values()) for k, v in out.items()}


def ghep_manh(manh, out, L):
    """Ghép các mảnh GEE vào ĐÚNG khung lưới L (nodata ngoài phần có mảnh). Kiểm: cỡ điểm ảnh, lệch nguyên điểm ảnh,
    đủ phủ khung. Trả về dict(du, so_manh, pct_phu)."""
    import rasterio
    from rasterio.windows import Window
    tf, W, H = transform_luoi(L), L["rong"], L["cao"]
    phu = np.zeros((H, W), bool)
    with rasterio.open(manh[0]) as d0:
        prof = dict(driver="GTiff", width=W, height=H, count=d0.count, dtype=d0.dtypes[0], crs=CRS, transform=tf,
                    nodata=NODATA, tiled=True, blockxsize=256, blockysize=256, compress="deflate", predictor=2,
                    BIGTIFF="IF_SAFER")
        mo_ta = d0.descriptions
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    with rasterio.open(out + ".part.tif", "w", **prof) as o:
        for p in manh:
            with rasterio.open(p) as d:
                if abs(d.transform.a - L["res"]) > 1e-6:
                    raise ValueError(f"{p}: cỡ điểm ảnh {d.transform.a} khác {L['res']}")
                fc, fr = (d.transform.c - tf.c) / L["res"], (tf.f - d.transform.f) / L["res"]
                if abs(fc - round(fc)) > 1e-6 or abs(fr - round(fr)) > 1e-6:
                    raise ValueError(f"{p}: lệch lưới ({fc:.3f}, {fr:.3f} điểm ảnh)")
                c0, r0 = int(round(fc)), int(round(fr))
                ca, ra = max(0, c0), max(0, r0)
                cb, rb = min(W, c0 + d.width), min(H, r0 + d.height)
                if cb <= ca or rb <= ra:
                    continue
                for rr in range(ra, rb, 1024):
                    h = min(1024, rb - rr)
                    a = d.read(window=Window(ca - c0, rr - r0, cb - ca, h))
                    o.write(a, window=Window(ca, rr, cb - ca, h))
                phu[ra:rb, ca:cb] = True
        if mo_ta and any(mo_ta):
            o.descriptions = tuple(m or f"b{i + 1}" for i, m in enumerate(mo_ta))
    os.replace(out + ".part.tif", out)
    pct = round(100.0 * phu.mean(), 2)
    return dict(du=bool(phu.all()), so_manh=len(manh), pct_phu=pct)


def mat_na_ranh(bnd, L):
    """Mặt nạ bool (cao, rộng) trên lưới L: True trong ranh giới (GeoDataFrame bất kỳ hệ toạ độ)."""
    from rasterio.features import geometry_mask
    g = bnd.to_crs(CRS)
    return geometry_mask(list(g.geometry), out_shape=(L["cao"], L["rong"]), transform=transform_luoi(L), invert=True)


def tham_chieu_trung_vi(ds_tep, out, rows=256):
    """Ảnh tham chiếu = trung vị từng điểm ảnh của nhiều ảnh cùng lưới (phương án A: các năm OLI 2015-2020, khi không có
    ảnh tham chiếu GEE). Ghi 10 băng như BANG_RA: 6 phản xạ trung vị, TEMP trung vị, NOBS = số ảnh có dữ liệu,
    NGUON 0, CAMBIEN = OR các ảnh."""
    import rasterio
    from rasterio.windows import Window
    srcs = [rasterio.open(t) for t in ds_tep]
    try:
        d0 = srcs[0]
        prof = d0.profile.copy()
        prof.update(count=len(BANG_RA), dtype="int16", nodata=NODATA, compress="deflate", predictor=2, tiled=True,
                    blockxsize=256, blockysize=256)
        with rasterio.open(out, "w", **prof) as o:
            for r0 in range(0, d0.height, rows):
                h = min(rows, d0.height - r0)
                win = Window(0, r0, d0.width, h)
                A = np.stack([s.read(window=win) for s in srcs]).astype(np.float64)       # (n, 10, h, w)
                hop = (A[:, :6] != NODATA).all(1)                                           # (n, h, w)
                X = np.where(hop[:, None], A[:, :7], np.nan)
                X[:, 6][A[:, 6] == NODATA] = np.nan
                import warnings
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    med = np.nanmedian(X, axis=0)
                ra = np.full((len(BANG_RA), h, d0.width), NODATA, np.int16)
                co = hop.any(0)
                ra[:7] = np.where(np.isfinite(med), np.round(med), NODATA).astype(np.int16)
                ra[:6, ~co] = NODATA
                ra[7] = np.where(co, hop.sum(0), NODATA)
                ra[8] = np.where(co, 0, NODATA)
                cb = np.zeros((h, d0.width), np.int64)
                for k in range(len(srcs)):
                    cb |= np.where(hop[k], np.maximum(A[k, 9], 0), 0).astype(np.int64)
                ra[9] = np.where(co, cb, NODATA)
                o.write(ra, window=win)
            o.descriptions = tuple(BANG_RA)
            o.update_tags(tham_chieu="trung vị " + ", ".join(os.path.basename(t) for t in ds_tep))
    finally:
        for s in srcs:
            s.close()
    return out

In [ ]:
%%writefile /content/ls_mod/ls_pca.py
# =============================================================================
# LANDSAT 1987-2026: PCA phổ CHUỖI NĂM với MỘT bộ hệ số chung cho mọi năm (như PCA của S2 trong pcats)
#   Đầu vào: ảnh mùa khô 30 m ĐÃ chuẩn hoá tương đối (ls_landsat: chuan_hoa_anh / ls_gee, rồi ap_chuan_hoa IR-MAD),
#            10 băng BLUE..SWIR2 (×10000), TEMP, NOBS, NGUON, CAMBIEN.
#   Đặc trưng mỗi điểm ảnh: 6 băng phản xạ + NDVI, MNDWI, NDBI (9 chiều).
#   Khớp: cùng một bộ điểm ảnh ngẫu nhiên cho mọi năm (như bộ điểm cố định của pcats), chỉ dùng điểm NGUON ≤ 1
#         (≥ 2 lần quan sát trong mùa), bỏ năm trong nam_khong_fit; chọn số PC bằng 7 tiêu chí không nhãn của
#         pca_ts_py.phan_tich_so_pc và (nếu có) độ chính xác phân loại 3 lớp tại điểm có nhãn (tiêu chí 8).
#   Vì sao KHÔNG làm PCA nhiều kỳ trong năm cho Landsat: trước 2013 chỉ có một hoặc hai cảm biến 16 ngày, mùa mưa ở
#   Hải Phòng gần như luôn có mây, nên kỳ mùa mưa phần lớn phải nội suy; PC khi đó phản ánh mật độ ảnh của từng thời
#   kỳ hơn là mặt đất. Ảnh mùa khô có ở mọi năm, đã chuẩn hoá chéo cảm biến, nên PC so được giữa 1987 và 2026.
#   (PCA nhiều kỳ Landsat bản cũ vẫn có trong pcats, cấu hình "LS", cho ai cần đường mùa vụ và chấp nhận NFILL cao.)
# Sản phẩm: lspc_{năm}.tif (k PC × 100, int16, lưới 30 m gốc S2) + ảnh xám COG 3857 từng PC cho geoportal.
# =============================================================================
import os, json, time
import numpy as np
import pandas as pd

from ls_landsat import TEN_BANG, NODATA, HE_SO, CRS

__version__ = "1.0"
DAC_TRUNG = TEN_BANG + ["NDVI", "MNDWI", "NDBI"]
CHI_SO = {"NDVI": ("NIR", "RED"), "MNDWI": ("GREEN", "SWIR1"), "NDBI": ("SWIR1", "NIR")}
CAU_HINH_LSPCA = dict(
    ten="Landsat mùa khô 30 m, đã chuẩn hoá tương đối (IR-MAD), PCA phổ chuỗi năm",
    ky=[[11, 4]], dac_trung=DAC_TRUNG,
    nam_khong_fit=[2012],          # mùa khô 2012 chỉ còn ETM+ SLC-off: khe dữ liệu lớn, không dùng để khớp
    nfill_max_fit=0,               # NFILL := 1 khi NGUON là 2 hoặc 3 (một lần quan sát, mượn năm lân cận)
    k_xet=9, k_geoportal=5, so_diem_moi_nam=6000, hat_giong=20261001)


def ten_cot(cfg=None):
    """Tên cột theo quy ước pca_ts_py (đặc trưng_p01): dùng lại được mọi hàm khớp và chọn k của pcats."""
    cfg = cfg or CAU_HINH_LSPCA
    return [f"{f}_p01" for f in cfg["dac_trung"]]


def dac_trung(sr):
    """sr: (6, ...) phản xạ (đơn vị 0..1, float). Trả về (9, ...): 6 băng + NDVI, MNDWI, NDBI.
    Chỉ số = (a - b) / (a + b), cắt [-1, 1]; mẫu số gần 0 cho 0 (như _nd của pcats, không che nước có phản xạ âm)."""
    sr = np.asarray(sr, np.float64)
    out = [sr[i] for i in range(6)]
    for ten, (a, b) in CHI_SO.items():
        x, y = sr[TEN_BANG.index(a)], sr[TEN_BANG.index(b)]
        s = x + y
        with np.errstate(all="ignore"):
            v = np.where(np.abs(s) > 1e-6, (x - y) / s, 0.0)
        out.append(np.clip(v, -1, 1))
    return np.stack(out)


# --------------------------------------------------------------------------- lấy mẫu
def chon_diem(tep_mau, n, seed, mask=None, toi_thieu_nam=None, tep_theo_nam=None):
    """Bộ điểm ảnh (hàng, cột) ngẫu nhiên CỐ ĐỊNH, dùng chung mọi năm (ảnh mọi năm cùng lưới).
    mask: mảng bool (ranh giới) cùng cỡ ảnh; None = mọi điểm có dữ liệu ở tep_mau."""
    import rasterio
    with rasterio.open(tep_mau) as d:
        H, W = d.height, d.width
        a = d.read(1)
    ok = (a != NODATA) if mask is None else (mask & (a != NODATA))
    rr, cc = np.nonzero(ok)
    rng = np.random.default_rng(seed)
    j = np.sort(rng.choice(len(rr), min(n, len(rr)), replace=False))
    return rr[j], cc[j], (H, W)


def doc_diem(tep, rr, cc):
    """Giá trị 10 băng tại các điểm ảnh (đọc theo khối hàng, không nạp cả ảnh)."""
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(tep) as d:
        out = np.full((d.count, len(rr)), NODATA, np.int32)
        o = np.argsort(rr)
        r_s, c_s = rr[o], cc[o]
        for r0 in range(0, d.height, 1024):
            sel = (r_s >= r0) & (r_s < r0 + 1024)
            if not sel.any():
                continue
            h = min(1024, d.height - r0)
            a = d.read(window=Window(0, r0, d.width, h))
            out[:, o[sel]] = a[:, r_s[sel] - r0, c_s[sel]]
    return out


def bang_mau(tep, rr, cc, nam):
    """DataFrame theo quy ước pcats: cột đặc trưng_p01, NOBS, NFILL, NGUON, CAMBIEN, system:index (mã điểm ảnh)."""
    a = doc_diem(tep, rr, cc)
    hop = (a[:6] != NODATA).all(0)
    sr = a[:6].astype(np.float64) / HE_SO
    f = dac_trung(sr)
    df = pd.DataFrame(f.T, columns=ten_cot())
    df.loc[~hop, :] = np.nan
    ng = a[8].astype(float); ng[a[8] == NODATA] = np.nan
    df["NOBS"] = np.where(a[7] == NODATA, np.nan, a[7])
    df["NGUON"] = ng
    df["NFILL"] = np.where(np.isnan(ng), 9, np.isin(ng, (2, 3)).astype(int))     # NGUON 9 (ảnh có sẵn, phương án A): dùng được
    df["CAMBIEN"] = np.where(a[9] == NODATA, 0, a[9])
    df["system:index"] = [f"r{r}c{c}" for r, c in zip(rr, cc)]
    df["nam"] = int(nam)
    return df


def lay_mau(tep_theo_nam, n=None, seed=None, mask=None, nam_mau=None):
    """{năm: tệp} -> {năm: DataFrame}, cùng bộ điểm ảnh. Bộ điểm rút từ năm nam_mau (mặc định năm có nhiều dữ liệu nhất)."""
    cfg = CAU_HINH_LSPCA
    n, seed = n or cfg["so_diem_moi_nam"], seed or cfg["hat_giong"]
    nam_mau = nam_mau or _nam_day_du(tep_theo_nam)
    rr, cc, _ = chon_diem(tep_theo_nam[nam_mau], n, seed, mask)
    out = {}
    for y, t in sorted(tep_theo_nam.items()):
        out[int(y)] = bang_mau(t, rr, cc, y)          # điểm thiếu dữ liệu ở năm y: NaN, NFILL = 9 (không dùng để khớp)
    return out


def _nam_day_du(tep_theo_nam):
    import rasterio
    tot, best = -1, None
    for y, t in tep_theo_nam.items():
        with rasterio.open(t) as d:
            k = max(1, min(d.width, d.height) // 400)
            a = d.read(1, out_shape=(d.height // k, d.width // k))
        v = float((a != NODATA).mean())
        if v > tot:
            tot, best = v, y
    return best


# --------------------------------------------------------------------------- khớp, chọn k (dùng lại pca_ts_py)
def khop(mau_theo_nam, P, cfg=None):
    """P: module pca_ts_py. Trả về hệ số ĐỦ mọi thành phần (chưa chốt k)."""
    cfg = cfg or CAU_HINH_LSPCA
    hs = P.khop_pca(mau_theo_nam, dict(cfg, ma_cau_hinh=ma_cau_hinh(cfg)), "LSPCA")
    hs["dac_trung"] = cfg["dac_trung"]
    hs["don_vi"] = "phản xạ 0..1 và chỉ số -1..1, chuẩn hoá z theo mean/std chung"
    return hs


def ma_cau_hinh(cfg=None):
    import hashlib
    cfg = cfg or CAU_HINH_LSPCA
    d = {k: cfg[k] for k in ("dac_trung", "nam_khong_fit", "nfill_max_fit")}
    return hashlib.md5(json.dumps(d, sort_keys=True).encode()).hexdigest()[:6]


def pc_tu_mau(df, hs):
    """PC (N, k) của DataFrame mẫu."""
    Z = (df[hs["bang"]].to_numpy(np.float64) - np.asarray(hs["mean"])) / np.asarray(hs["std"])
    return Z @ np.asarray(hs["W"]).T


def troi_theo_cam_bien(mau_theo_nam, hs):
    """Trung bình PC theo năm VÀ cảm biến chủ đạo (để thấy bước nhảy khi đổi cảm biến: 1999, 2013, 2022).
    Chỉ điểm có dữ liệu ở mọi năm; cột 'buoc_nhay_PCj' = chênh so với năm trước, chia độ lệch chuẩn PCj."""
    rows = []
    ten = hs["bang"]
    chung = None
    for y, df in mau_theo_nam.items():
        s = set(df.loc[df[ten].notna().all(1), "system:index"])
        chung = s if chung is None else chung & s
    sd = None
    for y, df in sorted(mau_theo_nam.items()):
        dd = df[df["system:index"].isin(chung)]
        if not len(dd):
            continue
        Pc = pc_tu_mau(dd, hs)
        if sd is None:
            sd = Pc.std(0) + 1e-9
        cb = int(np.bincount(dd.CAMBIEN.astype(int)).argmax())
        r = dict(nam=int(y), so_diem=len(dd), cam_bien_chu_dao=cb, ti_le_nguon_ge2=float((dd.NGUON >= 2).mean()))
        for j in range(min(5, Pc.shape[1])):
            r[f"PC{j + 1}_tb"] = float(Pc[:, j].mean())
        rows.append(r)
    t = pd.DataFrame(rows)
    for j in range(min(5, len(hs["W"]))):
        c = f"PC{j + 1}_tb"
        if c in t:
            t[f"buoc_nhay_PC{j + 1}"] = t[c].diff() / sd[j]
    return t


# --------------------------------------------------------------------------- điểm có nhãn (tiêu chí 8)
def mau_tai_diem_nhan(tep_theo_nam, diem_nhan):
    """diem_nhan: list dict (pid, lon, lat, nam, cls, nguon) như pca_ts_py.lap_diem_nhan. Đọc đặc trưng Landsat tại điểm
    trong đúng năm. Trả về DataFrame có cột đặc trưng_p01, cls, nguon, nam, lon, lat."""
    import rasterio
    from pyproj import Transformer
    d = pd.DataFrame(diem_nhan)
    tr = Transformer.from_crs(4326, CRS, always_xy=True)
    x, y = tr.transform(d.lon.values, d.lat.values)
    d["x"], d["y"] = x, y
    rows = []
    for nam, g in d.groupby("nam"):
        t = tep_theo_nam.get(int(nam))
        if t is None:
            continue
        with rasterio.open(t) as ds:
            r, c = rasterio.transform.rowcol(ds.transform, g.x.values, g.y.values)
            r, c = np.asarray(r), np.asarray(c)
            trong = (r >= 0) & (r < ds.height) & (c >= 0) & (c < ds.width)
        a = np.full((10, len(g)), NODATA)
        if trong.any():
            a[:, trong] = doc_diem(t, r[trong], c[trong])
        hop = (a[:6] != NODATA).all(0)
        f = dac_trung(a[:6].astype(float) / HE_SO)
        df = pd.DataFrame(f.T, columns=ten_cot())
        df.loc[~hop, :] = np.nan
        for k in ("cls", "nguon", "nam", "lon", "lat", "pid"):
            df[k] = g[k].values
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


# --------------------------------------------------------------------------- áp hệ số lên ảnh
def ap_anh(tep, out, hs, k=None, rows=512):
    """Ảnh mùa khô 10 băng -> k PC × he_so_nhan (int16), nodata -32768, CÙNG lưới. Thêm băng NGUON để biết độ tin."""
    import rasterio
    from rasterio.windows import Window
    k = int(k or hs["k"])
    W_ = np.asarray(hs["W"])[:k]
    mu, sd = np.asarray(hs["mean"]), np.asarray(hs["std"])
    nhan = float(hs.get("he_so_nhan", 100))
    with rasterio.open(tep) as d:
        prof = d.profile.copy()
        prof.update(count=k + 1, dtype="int16", nodata=NODATA, compress="deflate", predictor=2, tiled=True,
                    blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER")
        prof.pop("photometric", None)
        os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
        with rasterio.open(out, "w", **prof) as o:
            for r0 in range(0, d.height, rows):
                h = min(rows, d.height - r0)
                win = Window(0, r0, d.width, h)
                a = d.read(window=win)
                hop = (a[:6] != NODATA).all(0)
                f = dac_trung(a[:6].astype(np.float64) / HE_SO).reshape(len(DAC_TRUNG), -1)
                z = (f - mu[:, None]) / sd[:, None]
                pc = (W_ @ z).reshape(k, h, d.width)
                ra = np.clip(np.round(pc * nhan), -32767, 32767).astype(np.int16)
                ra[:, ~hop] = NODATA
                ng = a[8].astype(np.int16); ng[~hop] = NODATA
                o.write(np.concatenate([ra, ng[None]]), window=win)
            o.descriptions = tuple([f"PC{j:02d}" for j in range(1, k + 1)] + ["NGUON"])
            o.update_tags(ma_he_so=hs.get("ma", ""), he_so_nhan=str(nhan), dac_trung=",".join(DAC_TRUNG))
    return out


def keo_gian_pc(mau_theo_nam, hs, k=None, p=98):
    """Kéo giãn chung mọi năm cho ảnh xám PC (đỏ +, xanh −, trắng 0): [lo, hi] = [-m, +m], m = phân vị p của |PC|
    (đơn vị PC × he_so_nhan, như trường keo_gian của lớp pc1..5 của S2)."""
    k = int(k or hs["k"])
    X = np.vstack([pc_tu_mau(df[df[hs["bang"]].notna().all(1)], hs)[:, :k] for df in mau_theo_nam.values()])
    m = np.percentile(np.abs(X), p, axis=0) * float(hs.get("he_so_nhan", 100))
    return {q + 1: [int(-round(m[q])), int(round(m[q]))] for q in range(k)}


def muc_manifest_lspc(hs, nam, kg):
    """Mục 'lspc' của manifest: đủ để trang tính PC từ giá trị phản xạ tại điểm và vẽ đường năm."""
    return dict(duong_dan="lspc/lspc_{y}.tif", nam=sorted(int(y) for y in nam), k=int(hs["k"]),
                dac_trung=hs.get("dac_trung", DAC_TRUNG), mean=hs["mean"], std=hs["std"], W=hs["W"],
                ti_le=hs.get("ti_le", [])[:int(hs["k"])], he_so_nhan=hs.get("he_so_nhan", 100), keo_gian=kg,
                ma=hs.get("ma", ""), ly_do_chon_k=hs.get("ly_do_chon_k", ""), phien_ban_ma=__version__,
                ghi_chu="PCA phổ, một bộ hệ số cho mọi năm, trên ảnh mùa khô đã chuẩn hoá tương đối")

In [ ]:
%%writefile /content/ls_mod/ls_embed.py
# =============================================================================
# LANDSAT 1987-2026: embedding từng điểm ảnh có ngữ cảnh, cùng kiến trúc với g7 của S2
#   dilated_v4 (s2_pixlab.build_arch), rộng 48, giãn (1, 2, 4), sâu 3, 64 chiều, mục tiêu tái dựng che 60 %, 1500 bước.
#   Khác S2: 6 băng vào (BLUE..SWIR2 của ảnh mùa khô đã chuẩn hoá tương đối), 30 m.
#   Chuẩn hoá đầu vào bằng MỘT bộ trung bình, độ lệch chuẩn chung mọi năm (không theo từng ảnh): đổi thật của mặt đất
#   (đô thị sáng lên, ao hồ mở rộng) không bị chuẩn hoá mất, và năm nào cũng qua cùng một phép biến đổi.
#   Mảnh huấn luyện chia đều cho mọi năm (mỗi năm cùng số mảnh), nên 1987-2012 (TM, ETM+) nặng ngang 2013-2026 (OLI).
#   Tuỳ chọn chung_s2: thêm đầu ra 1 × 1 học dự đoán 6 thành phần chính của embedding g7 S2 (2017-2026, gộp 3 × 3 về
#   30 m, đọc từ chính các ảnh g7, g7b trên HF và phép chiếu emb/g7_phep_chieu.json). Khi đó phép chiếu của Landsat
#   là đầu ra này, nên màu lsg so được với màu g7 trên cùng một thang.
# Phần không cần torch (mảnh, mô men, đánh giá, phép chiếu, đọc mục tiêu S2) kiểm thử ở t_ls_embed.py;
# phần torch tự kiểm bằng self_test() trên Colab.
# =============================================================================
import os, json, time
import numpy as np
import pandas as pd

from ls_landsat import TEN_BANG, NODATA

__version__ = "1.0"
CAU_HINH_EMB = dict(
    ten="lsg7", arch="dilated_v4", in_ch=6, dim=64, width=48, dilations=(1, 2, 4), depth=3,
    objective="recon", steps=1500, bs=32, manh=128, tile=96, lr=1e-3, mask_ratio=0.6, seed=0,
    manh_moi_nam=48, pad=12,
    chung_s2=False, nam_s2=list(range(2017, 2027)), w_s2=1.0, n_comp=6)


# --------------------------------------------------------------------------- dữ liệu
def doc_6(tep):
    """6 băng phản xạ (int16 ×10000) và mặt nạ hợp lệ của một ảnh mùa khô 10 băng."""
    import rasterio
    with rasterio.open(tep) as d:
        a = d.read(list(range(1, 7)))
    return a, (a != NODATA).all(0)


def mo_men_chung(tep_theo_nam, n_moi_nam=20000, seed=0):
    """Trung bình, độ lệch chuẩn từng băng (đơn vị DN ×10000), cùng số điểm mỗi năm. Trả về (mean[6], std[6])."""
    rng = np.random.default_rng(seed)
    gom = []
    for y, t in sorted(tep_theo_nam.items()):
        a, ok = doc_6(t)
        rr, cc = np.nonzero(ok)
        if not len(rr):
            continue
        j = rng.choice(len(rr), min(n_moi_nam, len(rr)), replace=False)
        gom.append(a[:, rr[j], cc[j]].T)
    X = np.concatenate(gom).astype(np.float64)
    return X.mean(0).tolist(), np.maximum(X.std(0), 1.0).tolist()


def cat_manh(tep_theo_nam, n_moi_nam, P, seed=0, mask=None, buoc=None, them=None):
    """Mảnh P × P HỢP LỆ TOÀN BỘ (không lỗ, trong ranh giới nếu có mask), cùng số mảnh mỗi năm.
    them: {năm: mảng (C, H, W)} cắt kèm đúng vị trí (vd mục tiêu S2). Trả về (X int16 (N, 6, P, P), bảng vị trí, dict them)."""
    rng = np.random.default_rng(seed)
    buoc = buoc or max(8, P // 4)
    X, meta, T = [], [], {}
    for y, t in sorted(tep_theo_nam.items()):
        a, ok = doc_6(t)
        if mask is not None:
            ok = ok & mask
        H, W = ok.shape
        # đếm điểm hợp lệ trong cửa sổ bằng ảnh tích phân
        I = np.pad(ok.astype(np.int32).cumsum(0).cumsum(1), ((1, 0), (1, 0)))
        ung = []
        for r0 in range(0, H - P + 1, buoc):
            for c0 in range(0, W - P + 1, buoc):
                s = I[r0 + P, c0 + P] - I[r0, c0 + P] - I[r0 + P, c0] + I[r0, c0]
                if s == P * P:
                    ung.append((r0, c0))
        if not ung:
            continue
        pick = rng.choice(len(ung), min(n_moi_nam, len(ung)), replace=False)
        for i in pick:
            r0, c0 = ung[i]
            X.append(a[:, r0:r0 + P, c0:c0 + P])
            meta.append(dict(nam=int(y), r0=int(r0), c0=int(c0)))
            if them is not None:
                src = them.get(int(y))
                blk = (src[:, r0:r0 + P, c0:c0 + P] if src is not None
                       else np.full((next(iter(them.values())).shape[0], P, P), np.nan, np.float16))
                T.setdefault("T", []).append(blk.astype(np.float16))
    if not X:
        raise ValueError(f"không có mảnh {P} × {P} nào hợp lệ toàn bộ trong ranh giới: giảm 'manh' (EMB_CFG) hoặc kiểm tra ảnh")
    X = np.stack(X).astype(np.int16)
    T = np.stack(T["T"]) if T.get("T") else None
    return X, pd.DataFrame(meta), T


# --------------------------------------------------------------------------- mục tiêu S2 (tuỳ chọn chung_s2)
def muc_tieu_s2(tep_g, tep_gb, phep_chieu_s2, L, tmp_dir="/tmp"):
    """Đọc ảnh g7 (thành phần 1-3) và g7b (4-6) trên HF (COG 3857, 8 bit), gộp trung bình về lưới Landsat 30 m L,
    đổi 1..255 về giá trị thành phần theo lo, hi của phép chiếu S2. Trả về (6, H, W) float32, NaN nơi không có."""
    import rasterio
    from rasterio.enums import Resampling
    from rasterio.vrt import WarpedVRT
    from ls_landsat import transform_luoi, CRS
    tf, W, H = transform_luoi(L), L["rong"], L["cao"]
    lo, hi = np.asarray(phep_chieu_s2["lo"], np.float64), np.asarray(phep_chieu_s2["hi"], np.float64)
    out = np.full((6, H, W), np.nan, np.float32)
    for k, tep in enumerate((tep_g, tep_gb)):
        with rasterio.open(tep) as ds, WarpedVRT(ds, crs=CRS, transform=tf, width=W, height=H,
                                                 resampling=Resampling.average, src_nodata=0, nodata=0) as v:
            a = v.read([1, 2, 3]).astype(np.float64)
        m = (a == 0).any(0)
        for q in range(3):
            j = 3 * k + q
            x = lo[j] + (a[q] - 1) / 254.0 * (hi[j] - lo[j])
            x[m] = np.nan
            out[j] = x
    return out


def chuan_muc_tieu(T):
    """(N, 6, P, P) -> (T chuẩn hoá theo từng thành phần, tm, ts) bỏ qua NaN."""
    Tf = T.astype(np.float32)
    tm = np.nanmean(Tf, axis=(0, 2, 3)); ts = np.nanstd(Tf, axis=(0, 2, 3)) + 1e-6
    return ((Tf - tm[None, :, None, None]) / ts[None, :, None, None]).astype(np.float16), tm.tolist(), ts.tolist()


def dau_ra_thanh_phep_chieu(A, b, tm, ts, lo, hi):
    """Đầu ra 1 × 1 (y_chuẩn = A z + b) -> phép chiếu dạng fit_proj: y = C (z - mu) với y theo thang S2.
    y = ts * (A z + b) + tm  =>  C = ts[:, None] * A,  mu = -pinv(C) (ts * b + tm) (nghiệm chuẩn nhỏ nhất, đúng vì C đủ hạng)."""
    A, b, tm, ts = (np.asarray(v, np.float64) for v in (A, b, tm, ts))
    C = ts[:, None] * A
    mu = -np.linalg.pinv(C) @ (ts * b + tm)
    return dict(mu=mu.tolist(), comps=C.tolist(), lo=list(map(float, lo)), hi=list(map(float, hi)),
                kieu="dau_ra_chung_s2", ghi_chu="thành phần theo thang phép chiếu g7 của S2")


def dau_ra_ab(h):
    """Trọng số (A (6, dim), b (6,)) của đầu ra 1 × 1: nhận module torch hoặc state_dict đã lưu trong ckpt."""
    sd = h.state_dict() if hasattr(h, "state_dict") else h
    W = sd["weight"]; B = sd["bias"]
    W = W.detach().cpu().numpy() if hasattr(W, "detach") else np.asarray(W)
    B = B.detach().cpu().numpy() if hasattr(B, "detach") else np.asarray(B)
    return W.reshape(W.shape[0], W.shape[1]), B.reshape(-1)


def tim_lop_g7(man, ten="g7"):
    """Hai lớp embedding S2 (thành phần 1-3 và 4-6) và đường dẫn phép chiếu trong manifest. Bỏ lớp Landsat (nguon 'ls')."""
    import re as _re
    ds = [l for l in man.get("layers", []) if l.get("kieu") == "rgb" and l.get("phep_chieu") and l.get("nguon") != "ls"]
    nhom = {}
    for l in ds:
        m = _re.match(r"^Embedding (\S+?),", l.get("ten", ""))
        nhom.setdefault(m.group(1) if m else l["id"], []).append(l)
    if not nhom:
        return None
    ds = nhom.get(ten) or next(iter(nhom.values()))
    a = next((l for l in ds if "1-2-3" in l.get("ten", "")), None)
    b = next((l for l in ds if "4-5-6" in l.get("ten", "")), None)
    return (a, b, a["phep_chieu"]) if a and b else None


# --------------------------------------------------------------------------- đánh giá (numpy)
def r2_tuyen_tinh(Z, Y, ti_le_kiem=0.3, seed=0, alpha=1e-3):
    """Hồi quy ridge Z -> Y, R² trên phần để riêng, từng cột của Y. Z (N, d), Y (N, m)."""
    rng = np.random.default_rng(seed)
    ok = np.isfinite(Z).all(1) & np.isfinite(Y).all(1)
    Z, Y = Z[ok].astype(np.float64), Y[ok].astype(np.float64)
    p = rng.permutation(len(Z)); n_k = int(len(Z) * ti_le_kiem)
    k, h = p[:n_k], p[n_k:]
    zm, zs = Z[h].mean(0), Z[h].std(0) + 1e-9
    A = np.c_[(Z[h] - zm) / zs, np.ones(len(h))]
    B = np.linalg.solve(A.T @ A + alpha * np.eye(A.shape[1]), A.T @ Y[h])
    P_ = np.c_[(Z[k] - zm) / zs, np.ones(len(k))] @ B
    ss = ((Y[k] - Y[k].mean(0)) ** 2).sum(0)
    return (1 - ((Y[k] - P_) ** 2).sum(0) / np.maximum(ss, 1e-12)).tolist()


def on_dinh_theo_nam(Zn, Xn, nam, chuyen_cam_bien=(1999, 2013, 2022), q_on_dinh=0.25):
    """Zn: {năm: (N, d)} embedding tại CÙNG N điểm ảnh; Xn: {năm: (N, 6)} phản xạ chuẩn hoá z cùng điểm.
    Với mỗi cặp năm liền kề: điểm 'phổ ổn định' = |Δ phản xạ| thuộc q_on_dinh thấp nhất; đo trung vị |Δ embedding| (chuẩn
    hoá theo trung vị chung) trên điểm ổn định, và tương quan hạng Spearman giữa |Δ phổ| và |Δ embedding| trên mọi điểm.
    Bước chuyển cảm biến (1998->1999, 2012->2013, 2021->2022) được đánh dấu: trôi lớn bất thường ở đó là dấu hiệu lệch
    cảm biến lọt qua chuẩn hoá."""
    from scipy.stats import spearmanr
    ys = sorted(y for y in nam if y in Zn and y - 1 in Zn)
    dz_all = [np.linalg.norm(Zn[y] - Zn[y - 1], axis=1) for y in ys]
    thang = np.nanmedian(np.concatenate(dz_all)) if dz_all else 1.0
    rows = []
    for y, dz in zip(ys, dz_all):
        dx = np.linalg.norm(Xn[y] - Xn[y - 1], axis=1)
        ok = np.isfinite(dz) & np.isfinite(dx)
        if ok.sum() < 20:
            continue
        nguong = np.quantile(dx[ok], q_on_dinh)
        on = ok & (dx <= nguong)
        rows.append(dict(cap=f"{y - 1}-{y}", nam=y, so_diem=int(ok.sum()),
                         dz_on_dinh=float(np.median(dz[on]) / thang), dz_tat_ca=float(np.median(dz[ok]) / thang),
                         spearman=float(spearmanr(dx[ok], dz[ok]).correlation), chuyen_cam_bien=y in chuyen_cam_bien))
    t = pd.DataFrame(rows)
    if len(t):
        nen = t.loc[~t.chuyen_cam_bien, "dz_on_dinh"]
        t["ti_so_voi_nen"] = t.dz_on_dinh / (np.median(nen) if len(nen) else np.nan)
    return t


# --------------------------------------------------------------------------- torch
def _P():
    import s2_pixlab as P
    return P


def tao_encoder(mom, cfg=None, device="cuda"):
    cfg = dict(CAU_HINH_EMB, **(cfg or {}))
    P = _P()
    net = P.build_arch(cfg["arch"], cfg["in_ch"], cfg["dim"], width=cfg["width"], dilations=tuple(cfg["dilations"]),
                       depth=cfg["depth"])
    enc = P.DenseEncoder(cfg["ten"], net, mom[0], mom[1], cfg["dim"]).to(device)
    enc.pad = int(cfg["pad"])
    return enc


def huan_luyen(enc, X, cfg=None, T=None):
    """Không chung_s2: đúng train_dense của s2_pixlab (mục tiêu tái dựng che 60 %). Có chung_s2 (T khác None): vòng lặp
    riêng = tái dựng như trên + w_s2 × smooth L1 giữa đầu ra 1 × 1 (64 -> 6) và mục tiêu S2 đã chuẩn hoá, chỉ ở điểm có
    mục tiêu. Trả về (lịch sử, đầu ra 1 × 1 hoặc None)."""
    cfg = dict(CAU_HINH_EMB, **(cfg or {}))
    P = _P()
    if T is None or not cfg["chung_s2"]:
        return P.train_dense(enc, X, objective=cfg["objective"], steps=cfg["steps"], bs=cfg["bs"], tile=cfg["tile"],
                             lr=cfg["lr"], mask_ratio=cfg["mask_ratio"], seed=cfg["seed"]), None
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    dev = enc.device
    rng = np.random.default_rng(cfg["seed"]); torch.manual_seed(cfg["seed"])
    gen = torch.Generator(device=dev); gen.manual_seed(cfg["seed"])
    Xg = torch.from_numpy(np.clip(X, -32767, 32767).astype(np.int16)).to(dev)
    Tg = torch.from_numpy(np.asarray(T, np.float16)).to(dev)
    co_t = np.isfinite(np.asarray(T[:, 0, ::8, ::8], np.float32)).mean((1, 2)) > 0.5
    i_t, i_k = np.nonzero(co_t)[0], np.arange(len(X))
    dec = nn.Sequential(nn.Conv2d(enc.dim, 64, 3, padding=1), nn.GELU(), nn.Conv2d(64, X.shape[1], 1)).to(dev)
    head = nn.Conv2d(enc.dim, T.shape[1], 1).to(dev)
    params = list(enc.net.parameters()) + list(dec.parameters()) + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["lr"], total_steps=cfg["steps"], pct_start=0.1)
    Pp, tile, bs = X.shape[-1], cfg["tile"], cfg["bs"]
    hist, t0 = [], time.time()
    enc.net.train()
    for step in range(1, cfg["steps"] + 1):
        nt = bs // 2 if len(i_t) else 0                                    # nửa lô là mảnh có mục tiêu S2
        idx = np.r_[rng.choice(i_t, nt), rng.choice(i_k, bs - nt)].astype(int)
        y0 = rng.integers(0, Pp - tile + 1, bs); x0 = rng.integers(0, Pp - tile + 1, bs)
        xb = torch.stack([Xg[i, :, a:a + tile, b:b + tile] for i, a, b in zip(idx, y0, x0)])
        tb = torch.stack([Tg[i, :, a:a + tile, b:b + tile] for i, a, b in zip(idx, y0, x0)]).float()
        x = enc.norm(xb)
        mask = P.make_mask(bs, tile, tile, cfg["mask_ratio"], dev, gen)
        z_m = enc.net(x * (1 - mask))
        rec = (F.smooth_l1_loss(dec(z_m), x, reduction="none") * mask).sum() / (mask.sum() * x.shape[1] + 1e-6)
        z = enc.net(x)
        vt = torch.isfinite(tb).all(1, keepdim=True).float()
        dis = (F.smooth_l1_loss(head(z), torch.nan_to_num(tb), reduction="none") * vt).sum() / (vt.sum() * tb.shape[1] + 1e-6)
        loss = rec + cfg["w_s2"] * dis
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sch.step()
        hist.append(dict(step=step, loss=float(loss.detach()), rec=float(rec.detach()), s2=float(dis.detach())))
        if step % 100 == 0 or step == cfg["steps"]:
            print(f"    [{enc.name}/recon+s2] {step}/{cfg['steps']} loss {hist[-1]['loss']:.3f} rec {hist[-1]['rec']:.3f} "
                  f"s2 {hist[-1]['s2']:.3f} ({time.time() - t0:.0f}s)")
    enc.net.eval(); head.eval()
    return hist, head


def luu_ckpt(enc, path, cfg, mom, head=None, them=None):
    import torch
    cfg = dict(CAU_HINH_EMB, **(cfg or {}))
    st = dict(state_dict=enc.net.state_dict(), mean=list(mom[0]), std=list(mom[1]), pad=enc.pad,
              cfg={k: cfg[k] for k in ("arch", "in_ch", "dim", "width", "dilations", "depth", "objective", "steps",
                                       "chung_s2", "ten")},
              bang=TEN_BANG, chuan_hoa="chung mọi năm", phien_ban_ma=__version__, them=them or {})
    if head is not None:
        st["head"] = {k: v.detach().cpu() for k, v in head.state_dict().items()}
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save(st, path + ".part"); os.replace(path + ".part", path)
    return path


def nap_ckpt(path, device="cuda"):
    import torch
    st = torch.load(path, map_location="cpu", weights_only=False)
    c = st["cfg"]
    enc = tao_encoder((st["mean"], st["std"]), dict(c, pad=st.get("pad", 12)), device=device)
    enc.net.load_state_dict(st["state_dict"]); enc.net.eval()
    return enc, st


def embed_anh(enc, a6, ok, tile=1024, device=None, half=True):
    """(6, H, W) int16 + mặt nạ -> (dim, H, W) float16, NaN ngoài vùng hợp lệ. Chia ô có lề enc.pad, đệm đối xứng
    (như bản đồ g7). Điểm không hợp lệ được điền trung bình chung trước khi vào mạng để không làm méo hàng xóm.
    Lớp GRN chuẩn hoá theo cả ô nên kết quả phụ thuộc cỡ ô: lấy mẫu, phép chiếu và bản đồ đều dùng CÙNG tile (1024)."""
    import torch
    dev = device or enc.device
    pad = int(enc.pad)
    mean = np.asarray(enc.mean.detach().cpu()).reshape(-1)
    a = np.where(ok[None], a6, mean[:, None, None].astype(np.float32)).astype(np.float32)
    H, W = ok.shape
    out = np.full((enc.dim, H, W), np.nan, np.float16)
    use_amp = str(dev).startswith("cuda") and torch.cuda.is_available()
    enc.net.eval()
    for r0 in range(0, H, tile):
        for c0 in range(0, W, tile):
            th, tw = min(tile, H - r0), min(tile, W - c0)
            ra, rb, ca, cb = max(0, r0 - pad), min(H, r0 + th + pad), max(0, c0 - pad), min(W, c0 + tw + pad)
            blk = a[:, ra:rb, ca:cb]
            p_ = ((0, 0), (pad - (r0 - ra), pad - (rb - r0 - th)), (pad - (c0 - ca), pad - (cb - c0 - tw)))
            blk = np.pad(blk, p_, mode="symmetric")
            with torch.inference_mode():
                x = enc.norm(torch.from_numpy(blk[None]).to(dev))
                if use_amp:
                    with torch.autocast("cuda", dtype=torch.float16):
                        z = enc.net(x)
                else:
                    z = enc.net(x)
                z = z.float()[0, :, pad:pad + th, pad:pad + tw]
                if half:
                    z = z.half()
            out[:, r0:r0 + th, c0:c0 + tw] = z.cpu().numpy().astype(np.float16)
    out[:, ~ok] = np.nan
    return out


def mau_z(enc, tep_theo_nam, n_moi_nam=8000, seed=0, rr_cc=None):
    """Embedding tại điểm ảnh ngẫu nhiên, cùng số điểm mỗi năm (hoặc tại đúng rr_cc cho mọi năm).
    Trả về (Z (N, dim), X6 (N, 6) DN, mảng năm (N,))."""
    rng = np.random.default_rng(seed)
    Zs, Xs, Ys = [], [], []
    for y, t in sorted(tep_theo_nam.items()):
        a, ok = doc_6(t)
        z = embed_anh(enc, a, ok)
        if rr_cc is None:
            rr, cc = np.nonzero(ok)
            j = rng.choice(len(rr), min(n_moi_nam, len(rr)), replace=False)
            rr, cc = rr[j], cc[j]
        else:
            rr, cc = rr_cc
        Zs.append(z[:, rr, cc].T.astype(np.float32)); Xs.append(np.where(ok[rr, cc][:, None], a[:, rr, cc].T, np.nan))
        Ys.append(np.full(len(rr), int(y)))
    return np.concatenate(Zs), np.concatenate(Xs).astype(np.float32), np.concatenate(Ys)


def ghi_chieu(enc, tep, out, proj, tile=1024):
    """Ảnh chiếu uint8 n băng (1..255, 0 = trống) trên CÙNG lưới 30 m của ảnh vào (để dựng lsg, lsgb bằng rgb_cog_nhom)."""
    import rasterio
    from s2_hf_lop import quantize
    a, ok = doc_6(tep)
    z = embed_anh(enc, a, ok, tile=tile).astype(np.float32)
    mu, C = np.asarray(proj["mu"], np.float32), np.asarray(proj["comps"], np.float32)
    y = np.tensordot(C, z - mu[:, None, None], axes=(1, 0))
    q = quantize(y, proj["lo"], proj["hi"])
    q[:, ~(ok & np.isfinite(y).all(0))] = 0
    with rasterio.open(tep) as d:
        prof = dict(driver="GTiff", width=d.width, height=d.height, count=q.shape[0], dtype="uint8", crs=d.crs,
                    transform=d.transform, nodata=0, tiled=True, blockxsize=256, blockysize=256, compress="deflate")
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    with rasterio.open(out, "w", **prof) as o:
        o.write(q)
    return out, z


def self_test(device=None):
    """Chạy trên Colab (có torch): mảnh giả, huấn luyện vài bước cả hai chế độ, nhúng ảnh, phép chiếu đầu ra."""
    import torch
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    rng = np.random.default_rng(0)
    X = (rng.normal(1500, 400, (16, 6, 64, 64))).astype(np.int16)
    mom = (X.reshape(16, 6, -1).transpose(1, 0, 2).reshape(6, -1).mean(1).tolist(),
           X.reshape(16, 6, -1).transpose(1, 0, 2).reshape(6, -1).std(1).tolist())
    cfg = dict(steps=6, bs=4, tile=32, width=8, dim=16)
    enc = tao_encoder(mom, cfg, dev)
    h, _ = huan_luyen(enc, X, cfg)
    assert len(h) == 6 and np.isfinite(h[-1]["loss"])
    T = rng.normal(0, 1, (16, 6, 64, 64)).astype(np.float16); T[8:] = np.nan
    enc2 = tao_encoder(mom, cfg, dev)
    h2, head = huan_luyen(enc2, X, dict(cfg, chung_s2=True), T=T)
    assert head is not None and np.isfinite(h2[-1]["s2"])
    a6 = X[0, :, :50, :40]; ok = np.ones((50, 40), bool); ok[:3] = False
    z = embed_anh(enc2, a6, ok, tile=24)
    assert z.shape == (16, 50, 40) and np.isnan(z[:, 0]).all() and np.isfinite(z[:, 10]).all()
    z2 = embed_anh(enc2, a6, ok, tile=24)
    assert np.array_equal(np.nan_to_num(z), np.nan_to_num(z2)), "nhúng hai lần khác nhau"
    A = head.weight.detach().cpu().numpy()[:, :, 0, 0]; b = head.bias.detach().cpu().numpy()
    pj = dau_ra_thanh_phep_chieu(A, b, [0.0] * 6, [1.0] * 6, [-3] * 6, [3] * 6)
    zz = z[:, 10, 5].astype(np.float64)
    y1 = np.asarray(pj["comps"]) @ (zz - np.asarray(pj["mu"]))
    y2 = A @ zz + b
    assert np.allclose(y1, y2, atol=1e-4)
    print("ls_embed self_test: ĐẠT (", dev, ")")
    return True

In [ ]:
%%writefile /content/ls_mod/pca_ts_py.py
# =============================================================================
# PCA CHUỖI THỜI GIAN CHO HẢI PHÒNG MỚI, CHẠY TRÊN NHIỀU TÀI KHOẢN GEE
# Phần Python thuần (không cần Earth Engine): kế hoạch, chia việc, khớp PCA chung,
# gom và kiểm tra ảnh xuất.
# =============================================================================
import os, re, glob, json, math, time, hashlib
import numpy as np, pandas as pd

PHIEN_BAN = "v1"
GOC_LUOI = (615610.0, 2352020.0)       # gốc lưới của ảnh tổ hợp S2_HP_{năm} (EPSG:32648)
CRS = "EPSG:32648"

# Năm đã có mẫu: mẫu huấn luyện 2023-2026, điểm kiểm định 2017/2019/2021/2023/2025.
UU_TIEN_CHUNG = [2025, 2023, 2024, 2021, 2019, 2017, 2026, 2022, 2020, 2018]
# Năm có điểm mang nhãn: huấn luyện 2023-2026, kiểm định 2017/2019/2021/2023/2025.
NAM_CO_NHAN = [2017, 2019, 2021, 2023, 2024, 2025, 2026]

CAU_HINH_MAC_DINH = {
    "S2": dict(
        ten="Sentinel-2 L2A (S2_SR_HARMONIZED), mây lọc bằng Cloud Score+",
        nam=list(range(2017, 2027)), nam_tam=[2026], nam_khong_fit=[2017, 2026],
        uu_tien=UU_TIEN_CHUNG,
        ky=[[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]],      # 6 kỳ hai tháng
        dac_trung=["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12", "NDVI", "MNDWI"],
        res=10, k_xet=20, cs_min=0.60, may_canh_max=90, nfill_max_fit=2, so_diem_mau=5000),
    "S1": None,                                                        # xem S1_V2 bên dưới
    "LS": dict(
        ten="Landsat 5/7/8/9 Collection 2 L2 (không quy đổi chéo cảm biến)",
        nam=list(range(1987, 2027)), nam_tam=[2026], nam_khong_fit=[2012, 2026],
        uu_tien=UU_TIEN_CHUNG + [2015, 2013, 2010, 2005, 2000, 1995, 1990, 1987],
        ky=[[1, 3], [4, 6], [7, 9], [10, 12]],                         # 4 quý
        dac_trung=["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2", "NDVI", "MNDWI"],
        res=30, k_xet=20, may_canh_max=80, quy_doi_oli=False, dung_l7=True, nfill_max_fit=1,
        so_diem_mau=5000),
}

# S1 bản 2 (thay bản 1 sau khi bản 1 tốn khoảng 95 EECU-giờ cho 3 việc lấy mẫu mà vẫn hết giờ):
#  - 6 kỳ hai tháng (bản 1: 12 tháng), trùng kỳ với S2;
#  - MỘT hướng quỹ đạo, chốt ở bước 1b (tệp huong_s1.json), nên số cảnh giảm khoảng một nửa;
#  - 20 m: độ phân giải thật của GRD IW khoảng 20 x 22 m; lưới 20 m cùng gốc, mỗi ô = 2 x 2 ô 10 m;
#  - không nội suy từng điểm ảnh (S1 không bị mây che); kỳ trống cả vùng thì lấy trung vị cả năm;
#  - lọc trung vị 3 x 3 một lần trên chồng ảnh cuối, không lọc từng kỳ;
#  - NOBS là số cảnh trong năm của cả vùng (không đếm từng điểm ảnh để bớt một lượt đọc).
S1_V2 = dict(
    ten="Sentinel-1 GRD IW, VV và VH (dB), gamma0 theo góc tới, một hướng quỹ đạo",
    phien=2, nam=list(range(2015, 2027)), nam_tam=[2015, 2026], nam_khong_fit=[2015, 2026],
    uu_tien=UU_TIEN_CHUNG + [2016, 2015],
    ky=[[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]],
    dac_trung=["VV", "VH", "VVmVH"],
    res=20, k_xet=18, huong=None, gamma0=True, loc_khong_gian=True, noi_suy="ky",
    nfill_max_fit=1, so_diem_mau=3000)
CAU_HINH_MAC_DINH["S1"] = S1_V2
KHOA_MA_CAU_HINH = ("ky", "dac_trung", "res", "huong", "gamma0", "loc_khong_gian", "noi_suy",
                    "phien", "cs_min", "may_canh_max", "quy_doi_oli", "dung_l7")


def ma_cau_hinh(cfg):
    """Mã 6 ký tự của các tham số quyết định giá trị đầu vào PCA: ghi vào mỗi CSV mẫu để
    không bao giờ trộn CSV của hai cấu hình khác nhau."""
    d = {k: cfg.get(k) for k in KHOA_MA_CAU_HINH}
    return hashlib.md5(json.dumps(d, sort_keys=True).encode()).hexdigest()[:6]


def nang_cap_ke_hoach(ke_hoach, huong_s1=None):
    """
    Nâng kế hoạch đã lập (ke_hoach.json) lên cấu hình hiện hành, TRONG BỘ NHỚ, giống hệt nhau
    trên mọi tài khoản (không ghi đè tệp chung):
      - S1 bản 1 -> bản 2 (giữ nguyên năm, ưu tiên, việc và tên việc);
      - thêm lưới 20 m cùng gốc;
      - số điểm mẫu theo cảm biến (so_diem_mau) nếu chưa có;
      - hướng quỹ đạo S1 lấy từ huong_s1.json (None = chưa chốt: mọi việc S1 bị giữ lại).
    """
    cb = ke_hoach["cam_bien"]
    if cb["S1"].get("phien", 1) < 2:
        cu = cb["S1"]
        moi = json.loads(json.dumps(S1_V2))
        for k in ("nam", "nam_tam", "nam_khong_fit", "uu_tien"):
            moi[k] = cu[k]
        cb["S1"] = moi
    for ten, c in cb.items():
        c.setdefault("so_diem_mau", CAU_HINH_MAC_DINH[ten].get("so_diem_mau", ke_hoach["mau"]["so_diem"]))
        c["so_diem_mau"] = min(int(c["so_diem_mau"]), int(ke_hoach["mau"]["so_diem"]))
    if huong_s1 is not None:
        assert huong_s1 in ("ASCENDING", "DESCENDING", "CA_HAI"), huong_s1
        cb["S1"]["huong"] = huong_s1
    l10 = ke_hoach["luoi"]["10"]
    for r in sorted({c["res"] for c in cb.values()}):
        if str(r) not in ke_hoach["luoi"]:
            # khung 10 m đã phủ vùng xuất; nới ra mép lưới r m cùng gốc
            ke_hoach["luoi"][str(r)] = khung_luoi(l10["khung"], r, tuple(ke_hoach["goc_luoi"]))
    for c in cb.values():
        c["ma_cau_hinh"] = ma_cau_hinh(c)
    return ke_hoach


def ten_bang_dau_vao(cfg):
    """Tên các băng đầu vào PCA theo đúng thứ tự: kỳ trước, đặc trưng sau."""
    return [f"{f}_p{i + 1:02d}" for i in range(len(cfg["ky"])) for f in cfg["dac_trung"]]


def khung_luoi(bounds, res, goc=GOC_LUOI):
    """Khung chữ nhật phủ `bounds` (xmin, ymin, xmax, ymax, EPSG:32648) và nằm đúng mép
    điểm ảnh của lưới gốc: điểm ảnh 10 m và 30 m đều trùng khít lưới S2_HP_{năm}."""
    x0, y0 = goc
    xmin, ymin, xmax, ymax = bounds
    c0, c1 = math.floor((xmin - x0) / res), math.ceil((xmax - x0) / res)
    r0, r1 = math.floor((y0 - ymax) / res), math.ceil((y0 - ymin) / res)
    return dict(ct=[float(res), 0.0, x0, 0.0, -float(res), y0],
                khung=[x0 + c0 * res, y0 - r1 * res, x0 + c1 * res, y0 - r0 * res],
                rong=int(c1 - c0), cao=int(r1 - r0), cot0=int(c0), hang0=int(r0))


def lap_ke_hoach(bnd, tai_khoan, cau_hinh=None, dem_diem=15000, hat_giong=20260924,
                 dem_mau_dong=None, vien_m=500, don_gian_m=20):
    """
    bnd: GeoDataFrame ranh giới Hải Phòng mới (114 xã/phường).
    tai_khoan: danh sách nhãn tài khoản tham gia, theo thứ tự, vd ["tk1", "tk2", "tk3"].
    Trả về dict kế hoạch (ghi ra JSON để mọi tài khoản dùng chung).
    """
    from shapely.geometry import mapping
    cau_hinh = json.loads(json.dumps(cau_hinh or CAU_HINH_MAC_DINH))
    utm = bnd.to_crs(CRS)
    hp = utm.union_all() if hasattr(utm, "union_all") else utm.unary_union
    ranh = hp.simplify(don_gian_m, preserve_topology=True)
    vung = hp.buffer(vien_m).simplify(don_gian_m, preserve_topology=True)
    import geopandas as gpd
    g4326 = gpd.GeoSeries([ranh, vung], crs=CRS).to_crs(4326)
    luoi = {str(r): khung_luoi(vung.bounds, r) for r in sorted({c["res"] for c in cau_hinh.values()})}

    viec = []
    for giai_doan in ("MAU", "NHAN", "PCA"):
        viec += _tao_viec(cau_hinh, giai_doan, tai_khoan)
    return dict(
        phien_ban=PHIEN_BAN, tao_luc=time.strftime("%Y-%m-%d %H:%M"), crs=CRS,
        goc_luoi=list(GOC_LUOI), luoi=luoi,
        ranh_gioi_geojson=mapping(g4326.iloc[0]), vung_xuat_geojson=mapping(g4326.iloc[1]),
        dien_tich_km2=round(hp.area / 1e6, 1), vien_m=vien_m,
        tai_khoan=list(tai_khoan), cam_bien=cau_hinh,
        mau=dict(so_diem=int(dem_diem), hat_giong=int(hat_giong)),
        thu_muc_xuat="HP_PCA_TS_XUAT_{tk}", thu_muc_chung="HP_PCA_TS_CHUNG",
        cong_viec=viec)


def _tao_viec(cau_hinh, giai_doan, tai_khoan):
    ds = []
    for cb, c in cau_hinh.items():
        nam_cb = [y for y in c["nam"] if giai_doan != "NHAN" or y in NAM_CO_NHAN]
        thu_tu = [y for y in c["uu_tien"] if y in nam_cb] + \
                 [y for y in sorted(nam_cb, reverse=True) if y not in c["uu_tien"]]
        for k, y in enumerate(thu_tu):
            ds.append(dict(giai_doan=giai_doan, cam_bien=cb, nam=int(y), hang=k))
    # xen kẽ cảm biến theo hạng ưu tiên: S2 2025, S1 2025, LS 2025, S2 2023...
    ds.sort(key=lambda v: (v["hang"], ["S2", "S1", "LS"].index(v["cam_bien"])))
    for i, v in enumerate(ds):
        v["id"] = f"{giai_doan}_{v['cam_bien']}_{v['nam']}"
        v["tai_khoan"] = tai_khoan[i % len(tai_khoan)]
        v["tam"] = v["nam"] in cau_hinh[v["cam_bien"]]["nam_tam"]
    return ds


def bo_sung_viec_nhan(ke_hoach):
    """Kế hoạch lập bằng bản notebook cũ chưa có việc NHAN: thêm vào (không đổi việc cũ)."""
    if any(v["giai_doan"] == "NHAN" for v in ke_hoach["cong_viec"]):
        return False
    for c in ke_hoach["cam_bien"].values():                     # bản cũ dùng k_xuat
        c.setdefault("k_xet", 20)
    ke_hoach["cong_viec"] += _tao_viec(ke_hoach["cam_bien"], "NHAN", ke_hoach["tai_khoan"])
    return True


def viec_cua(ke_hoach, tk, giai_doan, lam_thay=(), da_xong=(), cam_bien=None):
    """
    Việc của tài khoản `tk`, cộng việc nhận thay: mỗi phần tử của `lam_thay` là nhãn tài khoản
    ("tk1": nhận mọi việc của tk1) hoặc mã việc cụ thể ("MAU_S2_2019"). Bỏ việc đã xong.
    `cam_bien`: chỉ lấy các cảm biến này (vd ["S2", "LS"]); None = mọi cảm biến.
    """
    ai = {x for x in lam_thay if x in ke_hoach["tai_khoan"]} | {tk}
    viec_le = {x for x in lam_thay if x not in ke_hoach["tai_khoan"]}
    xong = set(da_xong)
    return [v for v in ke_hoach["cong_viec"]
            if v["giai_doan"] == giai_doan and (v["tai_khoan"] in ai or v["id"] in viec_le)
            and v["id"] not in xong and (cam_bien is None or v["cam_bien"] in cam_bien)]


def ten_san_pham(cb, nam, ma_he_so, tam=False):
    return f"HP_PCA_{cb}_{nam}_{ma_he_so}" + ("_tam" if tam else "")


# ------------------------------------------------------------------ khớp PCA chung
def ma_bam(d):
    s = json.dumps({k: np.round(np.asarray(d[k], dtype=float), 6).tolist()
                    for k in ("mean", "std", "W")}, sort_keys=True)
    return hashlib.md5(s.encode()).hexdigest()[:6]


def doc_mau(duong_dan_csv, ten_dt, cfg=None):
    """Đọc CSV mẫu. Có `cfg` thì kiểm cột cfg_ma: CSV làm bằng cấu hình khác bị từ chối.
    S1 bản 2 bắt buộc có cột này (CSV S1 bản 1 có tên cột trùng một phần nhưng nghĩa khác)."""
    df = pd.read_csv(duong_dan_csv)
    ten = os.path.basename(duong_dan_csv)
    thieu = [c for c in ten_dt + ["NOBS", "NFILL"] if c not in df.columns]
    if thieu:
        raise ValueError(f"{ten} thiếu cột {thieu[:5]}")
    if cfg is not None:
        ma = cfg.get("ma_cau_hinh") or ma_cau_hinh(cfg)
        if "cfg_ma" in df.columns:
            co = set(df["cfg_ma"].astype(str))
            if co != {ma}:
                raise ValueError(f"{ten}: làm bằng cấu hình {sorted(co)}, cấu hình hiện hành {ma}: "
                                 "gửi lại việc này với LAM_LAI = True")
        elif cfg.get("phien", 1) >= 2:
            raise ValueError(f"{ten}: không có cột cfg_ma, tức là làm bằng cấu hình cũ: gửi lại")
    return df


def doc_mau_tot(ds_duong_dan, ten_dt, cfg=None):
    """Nhiều bản cùng tên (vd hai tài khoản cùng làm một việc): lấy bản đầu tiên hợp lệ.
    Trả về (DataFrame hoặc None, danh sách lỗi)."""
    loi = []
    for p in ds_duong_dan:
        try:
            return doc_mau(p, ten_dt, cfg), loi
        except ValueError as e:
            loi.append(str(e))
    return None, loi


def khop_pca(mau_theo_nam, cfg, cb):
    """
    mau_theo_nam: {năm: DataFrame mẫu} của MỘT cảm biến.
    Khớp PCA trên ma trận tương quan, gộp mọi năm khớp (cùng bộ điểm mỗi năm), bỏ điểm
    phải lấp quá `nfill_max_fit` kỳ. Trả về dict hệ số, dùng chung cho mọi năm.
    """
    ten = ten_bang_dau_vao(cfg)
    nam_fit = [y for y in sorted(mau_theo_nam) if y not in cfg["nam_khong_fit"]]
    if not nam_fit:
        raise ValueError(f"{cb}: chưa có năm nào dùng được để khớp")
    # cùng MỘT bộ điểm cho mọi năm: năm lấy 15.000 điểm và năm lấy 5.000 điểm đầu của cùng
    # dãy ngẫu nhiên thì chỉ giữ phần chung, để năm nào cũng nặng như nhau trong PCA gộp
    chung = None
    if all("system:index" in mau_theo_nam[y].columns for y in nam_fit):
        for y in nam_fit:
            s = set(mau_theo_nam[y]["system:index"].astype(str))
            chung = s if chung is None else chung & s
        nho = min(len(mau_theo_nam[y]) for y in nam_fit)
        if len(chung) < 0.5 * nho:
            raise ValueError(f"{cb}: các năm chỉ chung {len(chung)} điểm (năm ít nhất có {nho}): "
                             "bộ điểm không khớp giữa các tài khoản, kiểm tra hinh_gee.json")
    khoi, dem = [], {}
    for y in nam_fit:
        d = mau_theo_nam[y]
        if chung is not None:
            d = d[d["system:index"].astype(str).isin(chung)]
        d = d[(d.NFILL <= cfg["nfill_max_fit"]) & d[ten].notna().all(1)]
        dem[int(y)] = int(len(d))
        khoi.append(d[ten].to_numpy(np.float64))
    X = np.vstack(khoi)
    mu, sd = X.mean(0), X.std(0)
    sd = np.where(sd < 1e-9, 1.0, sd)
    Z = (X - mu) / sd
    C = np.cov(Z, rowvar=False)
    ev, V = np.linalg.eigh(C)
    thu_tu = np.argsort(ev)[::-1]
    ev, V = ev[thu_tu], V[:, thu_tu]
    V = V * np.sign(V[np.abs(V).argmax(0), np.arange(V.shape[1])])     # dấu cố định
    k = V.shape[1]                     # ĐỦ mọi thành phần; chot_k() cắt lại sau khi thử nghiệm
    kq = dict(cam_bien=cb, phien_ban=PHIEN_BAN, bang=ten, mean=mu.tolist(), std=sd.tolist(),
              W=V.T.tolist(), tri_rieng=np.clip(ev, 0, None).tolist(),
              ti_le=(np.clip(ev, 0, None) / np.clip(ev, 0, None).sum()).tolist(), k=k,
              he_so_nhan=100, da_chot=False,
              nam_fit=[int(y) for y in nam_fit], so_diem_fit=dem,
              nfill_max_fit=cfg["nfill_max_fit"], tao_luc=time.strftime("%Y-%m-%d %H:%M"),
              ma_cau_hinh=cfg.get("ma_cau_hinh") or ma_cau_hinh(cfg),
              so_diem_chung=len(chung) if chung is not None else None)
    kq["ma"] = ma_bam(kq)
    return kq


def giu_diem_chung(mau_theo_nam, cfg):
    """Giữ ở MỌI năm đúng phần điểm chung (theo system:index) của các năm dùng để khớp, để mọi
    tiêu chí ở bước 3b cũng tính trên cùng một bộ điểm. Trả về (dict mới, số điểm chung)."""
    nam_fit = [y for y in mau_theo_nam if y not in cfg["nam_khong_fit"]]
    if not nam_fit or not all("system:index" in mau_theo_nam[y].columns for y in mau_theo_nam):
        return mau_theo_nam, None
    chung = None
    for y in nam_fit:
        s = set(mau_theo_nam[y]["system:index"].astype(str))
        chung = s if chung is None else chung & s
    return ({y: d[d["system:index"].astype(str).isin(chung)].reset_index(drop=True)
             for y, d in mau_theo_nam.items()}, len(chung))


def diem_pc(df, he_so):
    ten = he_so["bang"]
    Z = (df[ten].to_numpy(np.float64) - np.asarray(he_so["mean"])) / np.asarray(he_so["std"])
    return Z @ np.asarray(he_so["W"]).T


def troi_theo_nam(mau_theo_nam, he_so):
    """
    Trung bình PC qua từng năm, tính trên các điểm có đủ dữ liệu ở MỌI năm (giao theo
    system:index), để so đúng cùng một bộ điểm. Năm nào lệch hẳn là năm có vấn đề ảnh.
    """
    ten = he_so["bang"]
    du = {y: d[d[ten].notna().all(1)] for y, d in mau_theo_nam.items()}
    chung = None
    if all("system:index" in d.columns for d in du.values()):
        for d in du.values():
            s = set(d["system:index"].astype(str))
            chung = s if chung is None else chung & s
    rows = []
    for y, d in sorted(du.items()):
        dd = d[d["system:index"].astype(str).isin(chung)] if chung else d
        P = diem_pc(dd, he_so)
        r = dict(nam=int(y), so_diem=len(dd), NOBS_tv=float(dd.NOBS.median()) if len(dd) else np.nan,
                 ti_le_co_lap=float((dd.NFILL > 0).mean()) if len(dd) else np.nan,
                 dung_de_khop=int(y) in he_so["nam_fit"])
        for j in range(min(4, P.shape[1])):
            r[f"PC{j + 1}_tb"] = float(P[:, j].mean()) if len(dd) else np.nan
        rows.append(r)
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ tìm thư mục
def tim_thu_muc(ten_mau, goc_drive="/content/drive"):
    """Tìm thư mục theo tên (có thể có ký tự đại diện) trong My Drive và các lối tắt."""
    ung = glob.glob(f"{goc_drive}/MyDrive/{ten_mau}") + \
          glob.glob(f"{goc_drive}/.shortcut-targets-by-id/*/{ten_mau}") + \
          glob.glob(f"{goc_drive}/MyDrive/*/{ten_mau}")
    return sorted({os.path.realpath(p): p for p in ung if os.path.isdir(p)}.values())


# ------------------------------------------------------------------ gom kết quả
RE_MANH = re.compile(r"^(.*?)(-\d{10}-\d{10})?$")


def tim_ket_qua(thu_muc_xuat):
    """Mọi CSV mẫu và mọi mảnh ảnh PCA trong các thư mục xuất của các tài khoản."""
    csv, tif = {}, {}
    for tm in thu_muc_xuat:
        for p in glob.glob(f"{tm}/MAU_*.csv") + glob.glob(f"{tm}/NHAN_*.csv"):
            csv.setdefault(os.path.splitext(os.path.basename(p))[0], []).append(p)
        for p in glob.glob(f"{tm}/HP_PCA_*.tif"):
            goc = RE_MANH.match(os.path.splitext(os.path.basename(p))[0]).group(1)
            tif.setdefault(goc, []).append(p)
    return csv, tif


def _khung_xuat(ke_hoach, cb):
    L = ke_hoach["luoi"][str(ke_hoach["cam_bien"][cb]["res"])]
    return L["ct"][0], L["khung"][0], L["khung"][3], int(L["rong"]), int(L["cao"])


def kiem_manh(manh, ke_hoach, cb):
    """
    Các mảnh GEE có phủ ĐỦ khung xuất không. GEE chia ảnh lớn thành nhiều mảnh
    (tên đuôi -hàng-cột); nếu ghép khi mới có một phần mảnh (việc chép/chuyển chưa xong,
    Drive chưa đồng bộ) thì ảnh ghép bị trống cả một vùng. Mảnh không chồng nhau, nên đủ
    khi tổng số điểm ảnh các mảnh bằng số điểm ảnh của khung và mép ngoài trùng khung.
    Trả về (đủ hay không, số mảnh, % khung đã có).
    """
    import rasterio
    res, x0, y1, W, H = _khung_xuat(ke_hoach, cb)
    tong, trai, tren, phai, duoi = 0, np.inf, -np.inf, -np.inf, np.inf
    for p in manh:
        with rasterio.open(p) as d:
            tong += d.width * d.height
            trai, tren = min(trai, d.bounds.left), max(tren, d.bounds.top)
            phai, duoi = max(phai, d.bounds.right), min(duoi, d.bounds.bottom)
    du = (tong == W * H and abs(trai - x0) < res / 2 and abs(tren - y1) < res / 2
          and round((phai - trai) / res) == W and round((tren - duoi) / res) == H)
    return bool(du), len(manh), round(100.0 * tong / (W * H), 1)


def kiem_tep_ghep(ra, ke_hoach, cb):
    """Tệp đã ghép có đúng kích thước và gốc của khung xuất không."""
    import rasterio
    res, x0, y1, W, H = _khung_xuat(ke_hoach, cb)
    try:
        with rasterio.open(ra) as d:
            return (d.width == W and d.height == H and abs(d.transform.c - x0) < res / 2
                    and abs(d.transform.f - y1) < res / 2)
    except Exception:                                               # noqa: BLE001
        return False


def ghep_va_kiem(manh, ra, ke_hoach, cb, tam_dir="/content/tmp_pca", k_thu_nho=4):
    """
    Ghép các mảnh GEE thành một tệp (chép theo dải 1024 hàng, không nạp cả ảnh vào RAM),
    kiểm lưới so với lưới gốc S2_HP, rồi thống kê trong ranh giới trên ảnh thu nhỏ.
    """
    import shutil, rasterio
    from rasterio.windows import Window
    from rasterio.transform import Affine
    from rasterio.enums import Resampling
    from rasterio.features import geometry_mask
    from shapely.geometry import shape
    import geopandas as gpd
    os.makedirs(tam_dir, exist_ok=True)
    cfg = ke_hoach["cam_bien"][cb]
    L = ke_hoach["luoi"][str(cfg["res"])]
    ds = [rasterio.open(p) for p in sorted(manh)]
    try:
        t0 = ds[0].transform
        res = t0.a
        trai, tren = min(d.bounds.left for d in ds), max(d.bounds.top for d in ds)
        phai, duoi = max(d.bounds.right for d in ds), min(d.bounds.bottom for d in ds)
        W, H = int(round((phai - trai) / res)), int(round((tren - duoi) / res))
        tf = Affine(res, 0, trai, 0, -res, tren)
        lech = ((trai - L["ct"][2]) / L["ct"][0], (tren - L["ct"][5]) / L["ct"][4])
        cung_luoi = (abs(res - L["ct"][0]) < 1e-9 and all(abs(v - round(v)) < 1e-6 for v in lech))
        prof = ds[0].profile.copy()
        prof.update(width=W, height=H, transform=tf, compress="deflate", predictor=2, tiled=True,
                    blockxsize=512, blockysize=512, BIGTIFF="YES")
        mo_ta = ds[0].descriptions
        tam = os.path.join(tam_dir, os.path.basename(ra))
        with rasterio.open(tam, "w", **prof) as o:
            for d in ds:
                c0 = int(round((d.bounds.left - trai) / res))
                r0 = int(round((tren - d.bounds.top) / res))
                for rr in range(0, d.height, 1024):
                    h = min(1024, d.height - rr)
                    o.write(d.read(window=Window(0, rr, d.width, h)),
                            window=Window(c0, r0 + rr, d.width, h))
            for i, m in enumerate(mo_ta, 1):
                if m:
                    o.set_band_description(i, m)
    finally:
        for d in ds:
            d.close()
    shutil.copy(tam, ra)
    # thống kê trong ranh giới, trên ảnh thu nhỏ k lần
    with rasterio.open(tam) as d:
        h, w = max(d.height // k_thu_nho, 1), max(d.width // k_thu_nho, 1)
        a = d.read(out_shape=(d.count, h, w), resampling=Resampling.nearest)
        tfk = d.transform * Affine.scale(d.width / w, d.height / h)
        nd = d.nodata
    os.remove(tam)
    rg = gpd.GeoSeries([shape(ke_hoach["ranh_gioi_geojson"])], crs=4326).to_crs(CRS)
    trong = geometry_mask(rg.geometry, out_shape=a.shape[1:], transform=tfk, invert=True)
    hop_le = trong & ((a[0] != nd) if nd is not None else True)
    ten_b = list(mo_ta)
    i_obs = ten_b.index("NOBS") if "NOBS" in ten_b else a.shape[0] - 2
    i_fill = ten_b.index("NFILL") if "NFILL" in ten_b else a.shape[0] - 1
    co = bool(hop_le.any())
    return dict(so_manh=len(manh), rong=W, cao=H, so_bang=int(a.shape[0]),
                lech_cot=round(lech[0], 3), lech_hang=round(lech[1], 3), cung_luoi=bool(cung_luoi),
                pct_co_du_lieu=round(100 * float(hop_le.sum()) / max(int(trong.sum()), 1), 2),
                NOBS_trung_vi=float(np.median(a[i_obs][hop_le])) if co else np.nan,
                pct_co_lap=round(100 * float((a[i_fill][hop_le] > 0).mean()), 2) if co else np.nan,
                NFILL_tb=round(float(a[i_fill][hop_le].mean()), 3) if co else np.nan)


def anh_xem_nhanh(p, k=8):
    """RGB từ PC1-3 (kéo giãn 2-98 %) để xem nhanh đường nối và vùng trống."""
    import rasterio
    from rasterio.enums import Resampling
    with rasterio.open(p) as d:
        h, w = max(d.height // k, 1), max(d.width // k, 1)
        a = d.read([1, 2, 3], out_shape=(3, h, w), resampling=Resampling.nearest).astype(np.float32)
        nd = d.nodata
    hl = (a[0] != nd) if nd is not None else np.ones(a.shape[1:], bool)
    rgb = np.zeros((h, w, 4), np.float32)
    for i in range(3):
        v = a[i][hl]
        lo, hi = (np.percentile(v, [2, 98]) if v.size else (0, 1))
        rgb[..., i] = np.clip((a[i] - lo) / max(hi - lo, 1e-6), 0, 1)
    rgb[..., 3] = hl
    return rgb


# ======================================================================================
# CHỌN SỐ PC BẰNG THỰC NGHIỆM
# ======================================================================================
def chot_k(he_so_day_du, k, ly_do=""):
    """Cắt hệ số về k thành phần đầu, đánh dấu đã chốt; mã băm đổi theo k."""
    h = json.loads(json.dumps(he_so_day_du))
    h["W"] = h["W"][:int(k)]
    h["k"] = int(k)
    h["da_chot"] = True
    h["ly_do_chon_k"] = ly_do
    h["ma"] = ma_bam(h)
    return h


def _pca_day_du(Z):
    ev, V = np.linalg.eigh(np.cov(Z, rowvar=False))
    o = np.argsort(ev)[::-1]
    ev, V = np.clip(ev[o], 0, None), V[:, o]
    V = V * np.sign(V[np.abs(V).argmax(0), np.arange(V.shape[1])])
    return ev, V                                      # cột j của V là PC j


def _giong_khong_gian(U, V, k):
    """Độ trùng hai không gian con k chiều: trung bình cos² các góc chính (1 là trùng)."""
    M = U[:, :k].T @ V[:, :k]
    return float((M ** 2).sum() / k)


def _dem_lien_tiep(dk):
    """Số phần tử đầu liên tiếp thoả điều kiện."""
    n = 0
    for x in dk:
        if not x:
            break
        n += 1
    return n


def phan_tich_so_pc(mau_theo_nam, cfg, k_xet=None, nguong=0.97, n_song_song=4000,
                    lap_song_song=30, lap_chia_doi=20, seed=0):
    """
    Bảy tiêu chí không cần nhãn, tính trên mẫu ngẫu nhiên các năm dùng để khớp:
      1. phương sai cộng dồn đạt 90/95/97/99 % (trong mẫu);
      2. Kaiser: trị riêng > 1 (PCA trên ma trận tương quan);
      3. gậy gãy (broken stick, Jackson 1993): PC j giữ lại khi tỉ lệ phương sai vượt
         kỳ vọng của một cây gậy bẻ ngẫu nhiên thành d đoạn;
      4. phân tích song song (Horn 1965): trị riêng phải vượt phân vị 95 % của trị riêng
         dữ liệu đã xáo từng cột (mất tương quan);
      5. ổn định khi chia đôi mẫu: PC j của hai nửa có hệ số Tucker >= 0.95;
      6. ổn định giữa các năm: PC j khớp riêng từng năm so với PC j khớp gộp, Tucker >= 0.95
         (trung vị qua các năm);
      7. phương sai giải thích trên NĂM ĐỂ RIÊNG: khớp trên các năm khác, đo trên năm để
         riêng (bỏ-một-năm), đạt `nguong`.
    Trả về (bảng theo k, dict số PC theo từng tiêu chí).
    """
    rng = np.random.default_rng(seed)
    ten = ten_bang_dau_vao(cfg)
    nam_fit = [y for y in sorted(mau_theo_nam) if y not in cfg["nam_khong_fit"]]
    khoi = {y: mau_theo_nam[y].loc[(mau_theo_nam[y].NFILL <= cfg["nfill_max_fit"])
                                   & mau_theo_nam[y][ten].notna().all(1), ten].to_numpy(np.float64)
            for y in nam_fit}
    X = np.vstack(list(khoi.values()))
    mu, sd = X.mean(0), np.where(X.std(0) < 1e-9, 1.0, X.std(0))
    Z = (X - mu) / sd
    d = Z.shape[1]
    K = min(k_xet or cfg.get("k_xet", 20), d)
    ev, V = _pca_day_du(Z)
    tl = ev / ev.sum()
    cd = np.cumsum(tl)

    gay = np.array([sum(1.0 / i for i in range(j, d + 1)) / d for j in range(1, d + 1)])

    idx = rng.choice(len(Z), min(n_song_song, len(Z)), replace=False)
    Zs = Z[idx]
    ev_s = np.sort(np.linalg.eigvalsh(np.corrcoef(Zs, rowvar=False)))[::-1]
    ngau = [np.sort(np.linalg.eigvalsh(np.corrcoef(
        np.column_stack([rng.permutation(Zs[:, j]) for j in range(d)]), rowvar=False)))[::-1]
        for _ in range(lap_song_song)]
    nguong_ss = np.percentile(np.array(ngau), 95, axis=0)

    sim_nua = np.zeros((lap_chia_doi, K)); tuck_nua = np.zeros((lap_chia_doi, K))
    for r in range(lap_chia_doi):
        p = rng.permutation(len(Z)); a, b = p[: len(p) // 2], p[len(p) // 2:]
        _, Va = _pca_day_du(Z[a]); _, Vb = _pca_day_du(Z[b])
        for k in range(1, K + 1):
            sim_nua[r, k - 1] = _giong_khong_gian(Va, Vb, k)
            tuck_nua[r, k - 1] = abs(float(Va[:, k - 1] @ Vb[:, k - 1]))

    sim_nam, tuck_nam = [], []
    for y, Xy in khoi.items():
        _, Vy = _pca_day_du((Xy - mu) / sd)
        sim_nam.append([_giong_khong_gian(V, Vy, k) for k in range(1, K + 1)])
        tuck_nam.append([abs(float(V[:, k - 1] @ Vy[:, k - 1])) for k in range(1, K + 1)])

    ngoai = []
    for y in nam_fit:
        Xo = np.vstack([khoi[t] for t in nam_fit if t != y])
        mo, so = Xo.mean(0), np.where(Xo.std(0) < 1e-9, 1.0, Xo.std(0))
        _, Vo = _pca_day_du((Xo - mo) / so)
        Zy = (khoi[y] - mo) / so
        tong = (Zy ** 2).sum()
        dong = []
        for k in range(1, K + 1):
            R = Zy - (Zy @ Vo[:, :k]) @ Vo[:, :k].T
            dong.append(1 - (R ** 2).sum() / tong)
        ngoai.append(dong)
    ngoai = np.array(ngoai)

    bang = pd.DataFrame(dict(
        k=np.arange(1, K + 1), tri_rieng=ev[:K], ti_le_pct=100 * tl[:K], cong_don_pct=100 * cd[:K],
        gay_gay_pct=100 * gay[:K], tri_rieng_mau_con=ev_s[:K], nguong_song_song=nguong_ss[:K],
        on_dinh_chia_doi=np.median(sim_nua, 0), tucker_chia_doi=np.median(tuck_nua, 0),
        on_dinh_giua_nam=np.median(np.array(sim_nam), 0), tucker_giua_nam=np.median(np.array(tuck_nam), 0),
        giai_thich_nam_de_rieng_pct=100 * ngoai.mean(0), giai_thich_nam_de_rieng_min_pct=100 * ngoai.min(0)))
    tieu_chi = {
        "phương sai 90 % (trong mẫu)": int(np.searchsorted(cd, 0.90) + 1),
        "phương sai 95 % (trong mẫu)": int(np.searchsorted(cd, 0.95) + 1),
        "phương sai 97 % (trong mẫu)": int(np.searchsorted(cd, 0.97) + 1),
        "phương sai 99 % (trong mẫu)": int(np.searchsorted(cd, 0.99) + 1),
        f"phương sai {int(100 * nguong)} % trên năm để riêng": int(np.searchsorted(ngoai.mean(0), nguong) + 1)
        if ngoai.mean(0).max() >= nguong else None,
        "Kaiser (trị riêng > 1)": int((ev > 1).sum()),
        "gậy gãy": _dem_lien_tiep(tl > gay),
        "phân tích song song (Horn)": _dem_lien_tiep(ev_s > nguong_ss),
        # Tucker từng PC (Lorenzo-Seva và ten Berge 2006: >= 0.95 coi là "như nhau"). Không
        # dùng độ trùng không gian con để CHỌN k: với mẫu lớn, cả hướng nhiễu cũng ổn định
        # và độ trùng trung bình luôn cao, nên nó chỉ in ra để tham khảo.
        "ổn định chia đôi (Tucker >= 0.95)": _dem_lien_tiep(np.median(tuck_nua, 0) >= 0.95),
        "ổn định giữa các năm (Tucker >= 0.95)": _dem_lien_tiep(np.median(np.array(tuck_nam), 0) >= 0.95),
    }
    return bang, tieu_chi


def danh_gia_phan_loai(nhan, he_so_day_du, k_xet=20, n_cay=200, lap_bs=500, seed=0, in_ra=True):
    """
    Tiêu chí 8, quyết định nhất: độ chính xác phân loại 3 lớp theo số PC.
    Học trên điểm huấn luyện (nguon == 'train', các năm 2023-2026), đo trên điểm kiểm định
    độc lập (nguon == 'val', 2017-2025, nhãn theo đúng năm). Kèm kiểm định chéo theo ô
    1 km trên tập huấn luyện. Mốc so sánh: dùng TẤT CẢ đặc trưng gốc (không PCA).
    Chọn k theo quy tắc một sai số chuẩn (Breiman và cs. 1984): k nhỏ nhất có độ chính xác
    không kém k tốt nhất quá một sai số chuẩn (bootstrap).
    """
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GroupKFold
    ten = he_so_day_du["bang"]
    d = nhan[nhan[ten].notna().all(1) & (nhan.cls > 0)].reset_index(drop=True)
    Z = (d[ten].to_numpy(np.float64) - np.asarray(he_so_day_du["mean"])) / np.asarray(he_so_day_du["std"])
    PC = Z @ np.asarray(he_so_day_du["W"]).T
    y = d.cls.to_numpy()
    tr, va = (d.nguon == "train").to_numpy(), (d.nguon == "val").to_numpy()
    nhom = (np.floor(d.lon / 0.01).astype(int) * 100000 + np.floor(d.lat / 0.01).astype(int)).to_numpy()
    rng = np.random.default_rng(seed)
    K = min(k_xet, PC.shape[1])
    rows = []
    for k in list(range(1, K + 1)) + ["tat_ca"]:
        F = Z if k == "tat_ca" else PC[:, :k]
        rf = RandomForestClassifier(n_cay, min_samples_leaf=2, class_weight="balanced_subsample",
                                    n_jobs=-1, random_state=seed)
        r = dict(k=k, so_dac_trung=F.shape[1])
        if tr.sum() and va.sum():
            rf.fit(F[tr], y[tr])
            dung = (rf.predict(F[va]) == y[va]).astype(float)
            bs = dung[rng.integers(0, len(dung), (lap_bs, len(dung)))].mean(1)
            r.update(OA_kiem_dinh=dung.mean(), SE_kiem_dinh=bs.std())
            for nm, g in d[va].groupby("nam"):
                r[f"OA_{nm}"] = dung[(d.nam[va] == nm).to_numpy()].mean()
        if tr.sum() >= 50:
            oa = []
            ntr = nhom[tr]
            for a, b in GroupKFold(5).split(F[tr], y[tr], ntr):
                rf.fit(F[tr][a], y[tr][a])
                oa.append((rf.predict(F[tr][b]) == y[tr][b]).mean())
            r.update(OA_cheo=float(np.mean(oa)), SE_cheo=float(np.std(oa) / np.sqrt(len(oa))))
        rows.append(r)
        if in_ra:
            print(f"    k = {k}: " + ", ".join(f"{c} {v:.4f}" for c, v in r.items()
                                               if c.startswith(("OA_kiem", "OA_cheo")) and v == v))
    bang = pd.DataFrame(rows)
    so = bang[bang.k != "tat_ca"].copy()
    cot, se = ("OA_kiem_dinh", "SE_kiem_dinh") if "OA_kiem_dinh" in so else ("OA_cheo", "SE_cheo")
    tot = so[cot].astype(float).idxmax()
    muc = so.loc[tot, cot] - so.loc[tot, se]
    k1se = int(so[so[cot] >= muc].k.astype(int).min())
    return bang, dict(k_mot_SE=k1se, k_tot_nhat=int(so.loc[tot, "k"]), thuoc_do=cot,
                      OA_tot_nhat=float(so.loc[tot, cot]), SE_tot_nhat=float(so.loc[tot, se]),
                      OA_tat_ca=float(bang[bang.k == "tat_ca"][cot].iloc[0]))


def de_xuat_k(tieu_chi, ket_qua_phan_loai=None):
    """
    Có nhãn: k theo quy tắc một sai số chuẩn trên độ chính xác phân loại (tiêu chí 8).
    Không có nhãn: trung vị của Kaiser, gậy gãy, phân tích song song, lấy số PC ổn định giữa
    các năm làm trần. Ngưỡng % phương sai KHÔNG dùng để chọn: sau khi chuẩn hoá, nhiễu dàn đều ra
    nhiều thành phần nhỏ nên ngưỡng 97 % phụ thuộc mức nhiễu hơn là số tín hiệu thật.
    """
    if ket_qua_phan_loai:
        return ket_qua_phan_loai["k_mot_SE"], (
            f"quy tắc một sai số chuẩn trên {ket_qua_phan_loai['thuoc_do']}: k tốt nhất "
            f"{ket_qua_phan_loai['k_tot_nhat']} ({ket_qua_phan_loai['OA_tot_nhat']:.4f} ± "
            f"{ket_qua_phan_loai['SE_tot_nhat']:.4f})")
    vung = [tieu_chi[t] for t in ("Kaiser (trị riêng > 1)", "gậy gãy", "phân tích song song (Horn)")
            if tieu_chi.get(t)]
    k = int(np.floor(np.median(vung))) if vung else 1
    tran = tieu_chi.get("ổn định giữa các năm (Tucker >= 0.95)") or k    # độ ổn định chỉ là trần
    return max(1, min(k, tran)), (f"trung vị của Kaiser, gậy gãy, song song {vung}, "
                                  f"không vượt số PC ổn định giữa các năm ({tran})")


def lap_diem_nhan(mau_huan_luyen, diem_kiem_dinh, nhan_kiem_dinh, cot_nam):
    """
    mau_huan_luyen: DataFrame samples3_clean (lon, lat, year, cls3, region).
    diem_kiem_dinh: GeoDataFrame E0 (id, geometry); nhan_kiem_dinh: DataFrame (id, cột nhãn).
    cot_nam: {năm: tên cột nhãn}. Nhãn 0 (không rõ) bị bỏ.
    """
    rows = []
    t = mau_huan_luyen
    if "region" in t.columns:
        t = t[t.region == "HP"]
    for i, r in enumerate(t.itertuples()):
        rows.append(dict(pid=f"t{i}", lon=float(r.lon), lat=float(r.lat), nam=int(r.year),
                         cls=int(r.cls3), nguon="train"))
    g = diem_kiem_dinh.to_crs(4326).merge(nhan_kiem_dinh, on="id", suffixes=("", "_nhan"))
    for nam, cot in cot_nam.items():
        for r in g.itertuples():
            c = int(getattr(r, cot))
            if c > 0:
                rows.append(dict(pid=f"v{r.id}_{nam}", lon=float(r.geometry.x), lat=float(r.geometry.y),
                                 nam=int(nam), cls=c, nguon="val"))
    return rows


# ------------------------------------------------------------------ chi phí EECU
HAN_MUC_CONG_DONG = 150.0          # EECU-giờ mỗi tháng mỗi dự án, bậc Community (từ 27/04/2026)
HAN_MUC_DONG_GOP = 1000.0          # bậc Contributor


def dien_tich_xuat_km2(ke_hoach):
    """Diện tích vùng xuất (ranh giới nới 500 m), km²."""
    import geopandas as gpd
    from shapely.geometry import shape
    g = gpd.GeoSeries([shape(ke_hoach["vung_xuat_geojson"])], crs=4326).to_crs(CRS)
    return float(g.area.iloc[0] / 1e6)


def du_bao_chi_phi(ke_hoach, do, so_nhan_theo_nam=None):
    """
    do: {cb: {"diem": (eecu_giay, so_diem), "anh": (eecu_giay, km2)}} đo từ các việc THU_*.
    Ngoại suy tuyến tính: chi phí lấy mẫu tỉ lệ số điểm, chi phí xuất ảnh tỉ lệ diện tích.
    Chỉ là ước lượng cỡ độ lớn (GEE không tính tiền tuyến tính hoàn toàn), đủ để biết có vừa
    hạn mức hay không. Trả về DataFrame, đơn vị EECU-giờ.
    """
    S = dien_tich_xuat_km2(ke_hoach)
    so_nhan_theo_nam = so_nhan_theo_nam or {}
    rows = []
    for cb, cfg in ke_hoach["cam_bien"].items():
        m = do.get(cb, {})
        viec = [v for v in ke_hoach["cong_viec"] if v["cam_bien"] == cb]
        n = {g: sum(v["giai_doan"] == g for v in viec) for g in ("MAU", "NHAN", "PCA")}
        r = dict(cam_bien=cb, so_viec_MAU=n["MAU"], so_viec_NHAN=n["NHAN"], so_viec_PCA=n["PCA"])
        if "diem" in m:
            e, k = m["diem"]
            mot = e / max(k, 1) / 3600.0                                  # EECU-giờ mỗi điểm
            r["MAU_moi_nam_h"] = mot * cfg["so_diem_mau"]
            nh = [so_nhan_theo_nam.get(int(v["nam"]), 0) for v in viec if v["giai_doan"] == "NHAN"]
            r["NHAN_tong_h"] = mot * sum(nh)
            r["MAU_tong_h"] = r["MAU_moi_nam_h"] * n["MAU"]
        if "anh" in m:
            e, a = m["anh"]
            r["PCA_moi_nam_h"] = e / max(a, 1e-9) * S / 3600.0
            r["PCA_tong_h"] = r["PCA_moi_nam_h"] * n["PCA"]
        co = [r[k] for k in ("MAU_tong_h", "NHAN_tong_h", "PCA_tong_h") if k in r]
        r["da_do"] = "+".join(k for k in ("diem", "anh") if k in m) or "chưa đo"
        r["tong_h"] = float(sum(co)) if co else float("nan")        # thiếu phần nào thì tổng còn thiếu
        r["so_thang_cong_dong"] = r["tong_h"] / HAN_MUC_CONG_DONG
        r["so_thang_dong_gop"] = r["tong_h"] / HAN_MUC_DONG_GOP
        rows.append(r)
    return pd.DataFrame(rows)

In [ ]:
# ======================= 1. NẠP MÔ-ĐUN, RANH GIỚI, LƯỚI, KẾ HOẠCH GEE =======================
for m in ("ls_landsat", "ls_pca", "ls_embed", "pca_ts_py", "s2_hf_lop"):
    sys.modules.pop(m, None)
import ls_landsat as LS, ls_pca as LP, ls_embed as LE, pca_ts_py as P, s2_hf_lop as H
print("ls_landsat", LS.__version__, "| ls_pca", LP.__version__, "| ls_embed", LE.__version__, "| s2_hf_lop", H.__version__)
ZIP_HP = f"{D}/TÀI LIỆU KHOÁ HỌC VIỄN THÁM/DỮ LIỆU/Hải Phòng (phường xã) - 34.zip"
bnd = None
try:
    import s2_multiyear as M
    bnd, _ = M.read_boundary(ZIP_HP)
except Exception as e:                                                   # noqa: BLE001
    print("không dùng được s2_multiyear.read_boundary:", e)
if bnd is None:
    import zipfile
    zipfile.ZipFile(ZIP_HP).extractall("/content/hp_bnd")
    bnd = gpd.read_file(sorted(glob.glob("/content/hp_bnd/**/*.shp", recursive=True))[0])
L = LS.luoi_tu_ranh(bnd)
MASK = LS.mat_na_ranh(bnd, L)
G3857 = H.grid_3857(L["khung"], "EPSG:32648", res=30.0)
print(f"lưới 30 m: {L['rong']} × {L['cao']} điểm ảnh, gốc lưới S2 {LS.GOC_LUOI}, trong ranh giới {MASK.mean():.1%}")
cho = P.tim_thu_muc("HP_LS_CHUNG")
if not cho:
    os.makedirs(f"{D}/HP_LS_CHUNG", exist_ok=True)
    cho = [f"{D}/HP_LS_CHUNG"]
CHUNG = cho[0]
p_kh = f"{CHUNG}/ke_hoach_ls.json"
if os.path.exists(p_kh) and not GHI_DE_KE_HOACH:
    KHL = json.load(open(p_kh, encoding="utf-8"))
    print("đã có kế hoạch GEE, mã cấu hình", KHL["ma_cau_hinh"], "(giữ nguyên)")
else:
    KHL = LS.lap_ke_hoach_gee(bnd, DANH_SACH_TK, NAM)
    json.dump(KHL, open(p_kh, "w", encoding="utf-8"), ensure_ascii=False)
    print("đã ghi", p_kh)
assert KHL["L"] == L, "lưới của kế hoạch khác lưới hiện tại: đặt GHI_DE_KE_HOACH = True rồi gửi lại việc GEE"
for tk, ys in KHL["chia"].items():
    print(f"  {tk}: {len(ys)} năm, đầu tiên {ys[:5]}")
print("Thư mục chung:", CHUNG, "-> chia sẻ cho các tài khoản phụ (Người chỉnh sửa).")

In [ ]:
# ======================= 2. KIỂM KÊ ẢNH CÓ SẴN (phương án A) =======================
if THU_MUC_A is None:
    c = []
    for sau in ("*", "*/*", "*/*/*", "*/*/*/*"):                      # tìm nông trước: Drive lớn thì ** rất lâu
        c = sorted(glob.glob(f"{D}/{sau}/{MAU_A}"))
        if c:
            break
    THU_MUC_A = os.path.dirname(c[0]) if c else None
print("thư mục ảnh có sẵn:", THU_MUC_A)
KK = None
if LAM["kiem_ke"] and THU_MUC_A:
    bc = sorted(glob.glob(f"{THU_MUC_A}/**/HP_Mosaic_Images_Report*.csv", recursive=True))
    KK = LS.kiem_ke(THU_MUC_A, ranh=bnd, mau=MAU_A, bao_cao_gee=bc[0] if bc else None, in_ra=False)
    KK.to_csv(f"{KQ}/kiem_ke_A.csv", index=False)
    print("năm thiếu:", LS.thieu_nam(KK, NAM[0], NAM[-1]))
    cot = [c for c in ["tep", "nam", "crs", "res_m", "phu_du_lieu_pct", "TEMP_p50", "NOBS_p50", "canh_mua_kho_may50",
                       "canh_bao"] if c in KK]
    display(KK[cot])

In [ ]:
# ======================= 3. ẢNH THÔ TRÊN LƯỚI 30 m: phương án A (chuẩn hoá ảnh có sẵn) hoặc B (gom GEE) =======================
RAW, TC, BGOM = {}, None, []
os.makedirs(f"{KQ}/{PHUONG_AN}/raw", exist_ok=True)
if PHUONG_AN == "A":
    assert THU_MUC_A, "không thấy ảnh có sẵn: đặt THU_MUC_A"
    for t in sorted(glob.glob(f"{THU_MUC_A}/**/{MAU_A}", recursive=True)):
        y = LS.nhan_dang_ten(t).get("nam")
        if y not in NAM:
            continue
        out = f"{KQ}/A/raw/ls_{y}.tif"
        if LAM_LAI or not H.hop_le(out):
            LS.chuan_hoa_anh(t, out, L, y, tmp_dir=TMP)
        RAW[y] = out
    tc_nam = [y for y in range(2015, 2021) if y in RAW]
    TC = f"{KQ}/A/raw/tham_chieu_{tc_nam[0]}_{tc_nam[-1]}.tif" if tc_nam else None
    if TC and (LAM_LAI or not H.hop_le(TC)):
        LS.tham_chieu_trung_vi([RAW[y] for y in tc_nam], TC)
    with rasterio.open(RAW[min(RAW)]) as d_:
        phu = float(((d_.read(1) != LS.NODATA) & MASK).sum() / MASK.sum())
    if phu < 0.95:
        print(f"CHÚ Ý: ảnh có sẵn chỉ phủ {phu:.0%} ranh giới Hải Phòng mới (ảnh cũ chỉ có Đông Hải Phòng)")
else:
    tx = LS.tim_xuat_gee([f"{CHUNG}/xuat"], ma=KHL["ma_cau_hinh"])
    for goc, manh in sorted(tx.items()):
        out = f"{KQ}/B/raw/{goc}.tif"
        if not LAM_LAI and H.hop_le(out):
            r = dict(du=True, so_manh=len(manh), pct_phu=100.0, da_co=True)
        else:
            r = LS.ghep_manh(manh, out, L)
            if not r["du"]:
                os.remove(out)
        BGOM.append(dict(tep=goc, **r))
        if r["du"]:
            m = re.match(r"LS_HP_(\d{4})_", goc)
            if m:
                RAW[int(m.group(1))] = out
            elif goc.startswith("LS_HP_TC_"):
                TC = out
    if BGOM:
        display(pd.DataFrame(BGOM))
    print("chưa có (việc GEE chưa xong hoặc chưa chạy ô CHÉP):", [y for y in NAM if y not in RAW])
print(f"ảnh thô: {len(RAW)} năm; ảnh tham chiếu: {TC}")

In [ ]:
# ======================= 4. CHUẨN HOÁ TƯƠNG ĐỐI (IR-MAD) VỀ ẢNH THAM CHIẾU CHUNG =======================
# Mỗi năm: IR-MAD (Canty và Nielsen 2008) chọn điểm ảnh bất biến so với ảnh tham chiếu, hồi quy trực giao từng băng
# (y = a x + b) khi r² ≥ 0.85 và hệ số góc trong [0.7, 1.43]; băng không đạt thì chỉ sửa độ lợi (tỉ lệ trung bình) nếu
# r² ≥ 0.5, không nữa thì giữ nguyên băng đó. Năm có quá ít điểm bất biến: giữ ảnh thô và ghi lý do (cần xem lại).
NORM, BCH = {}, []
assert TC and os.path.exists(TC), "chưa có ảnh tham chiếu (phương án B: tài khoản đầu tiên gửi kèm việc tham chiếu)"
os.makedirs(f"{KQ}/{PHUONG_AN}/norm", exist_ok=True)
for y, t in sorted(RAW.items()):
    out, p_j = f"{KQ}/{PHUONG_AN}/norm/ls_{y}.tif", f"{KQ}/{PHUONG_AN}/norm/ls_{y}.json"
    if not LAM_LAI and H.hop_le(out) and os.path.exists(p_j):
        hs = json.load(open(p_j))
    else:
        X, Y = LS.doc_cap(t, TC)
        hs = LS.he_so_chuan_hoa(X, Y)
        hs["ap_dung"] = bool(hs["dat"] and LAM["chuan_hoa"])
        if hs["ap_dung"]:
            LS.ap_chuan_hoa(t, out, hs)
        else:
            shutil.copy(t, out)
        json.dump(hs, open(p_j, "w"))
    NORM[y] = out
    with rasterio.open(out) as d_:
        cb = d_.read(10, out_shape=(max(1, d_.height // 8), max(1, d_.width // 8)))
    cbv = cb[cb > 0]
    BCH.append(dict(nam=y, cam_bien=int(np.bincount(cbv).argmax()) if cbv.size else 0, ap_dung=hs.get("ap_dung", False),
                    n_pif=hs["n_pif"], r2_min=min(hs["r2"]), a_RED=hs["a"][2], b_RED=hs["b"][2], a_NIR=hs["a"][3],
                    b_NIR=hs["b"][3], cach=",".join(hs.get("cach", [])), ly_do=hs.get("ly_do", "")))
BCH = pd.DataFrame(BCH); BCH.to_csv(f"{KQ}/{PHUONG_AN}/chuan_hoa.csv", index=False)
display(BCH)
print("năm không qua cổng (giữ ảnh thô):", BCH.loc[~BCH.ap_dung, "nam"].tolist())

In [ ]:
# ======================= 5. CẦU NỐI VỚI SENTINEL-2 (2017-2026, chỉ chẩn đoán) =======================
# Landsat 30 m so với S2 10 m gộp 3 × 3 trên cùng lưới, cùng mùa khô: hệ số OLS, r², độ lệch, RMSE (đơn vị phản xạ).
CN = {}
if LAM["cau_noi"]:
    for y in sorted(NORM):
        s2 = f"{DRIVE_MY}/S2_HP_{y}_v2.tif" if os.path.exists(f"{DRIVE_MY}/S2_HP_{y}_v2.tif") else f"{DRIVE_MY}/S2_HP_{y}.tif"
        if y < 2017 or not os.path.exists(s2):
            continue
        try:
            CN[y] = LS.cau_noi_s2(NORM[y], s2)
        except ValueError as e:
            print(y, e)
if CN:
    BCN = pd.DataFrame([dict(nam=y, bang=b, **v) for y, d in CN.items() for b, v in d.items()])
    BCN.to_csv(f"{KQ}/{PHUONG_AN}/cau_noi_s2.csv", index=False)
    display(BCN.groupby("bang")[["a", "b", "r2", "lech", "rmse"]].median().round(4))

In [ ]:
# ======================= 6. COG 10 BĂNG VÀ ẢNH MÀU THẬT (lstc) =======================
p_kg = f"{KQ}/{PHUONG_AN}/keo_gian.json"
if os.path.exists(p_kg) and not LAM_LAI:
    KG = json.load(open(p_kg))                       # kéo giãn cố định: thêm năm sau vẫn cùng thang màu
else:
    KG = LS.keo_gian(list(NORM.values())); json.dump(KG, open(p_kg, "w"))
print("kéo giãn chung (DN):", KG)
if LAM["cog"]:
    for y, t in sorted(NORM.items()):
        p_ls, p_tc = f"{HF_LS}/ls/ls_{y}.tif", f"{HF_LS}/lstc/lstc_{y}.tif"
        if LAM_LAI or not H.hop_le(p_ls):
            LS.lam_cog(t, p_ls)
        if LAM_LAI or not H.hop_le(p_tc):
            LS.anh_xem_nhanh(t, p_tc, G3857, KG, tmp_dir=TMP, verbose=False)
    print("COG:", len(glob.glob(f"{HF_LS}/ls/ls_*.tif")), "năm | lstc:", len(glob.glob(f"{HF_LS}/lstc/lstc_*.tif")), "năm")

In [ ]:
# ======================= 7. PCA PHỔ CHUỖI NĂM (một bộ hệ số cho mọi năm) =======================
HS5, KG_PC = None, None
if LAM["pca"]:
    cfg = dict(LP.CAU_HINH_LSPCA)
    p_hs = f"{KQ}/{PHUONG_AN}/he_so_lspca.json"
    MAU = LP.lay_mau(NORM, mask=MASK)
    if os.path.exists(p_hs) and not LAM_LAI:
        HS5 = json.load(open(p_hs))
        print("đã có hệ số", HS5["ma"], "| k =", HS5["k"], "|", HS5.get("ly_do_chon_k", ""))
    else:
        HS = LP.khop(MAU, P, cfg)
        BK, TCK = P.phan_tich_so_pc(MAU, cfg, k_xet=cfg["k_xet"])
        display(BK.round(3)); print(TCK)
        KQPL = None
        cho_dn = P.tim_thu_muc("HP_PCA_TS_CHUNG")
        p_dn = f"{cho_dn[0]}/diem_nhan.json" if cho_dn else None
        if p_dn and os.path.exists(p_dn):
            TDN = LP.mau_tai_diem_nhan(NORM, json.load(open(p_dn)))
            if len(TDN):
                BPL, KQPL = P.danh_gia_phan_loai(TDN, HS, k_xet=cfg["k_xet"])
                display(BPL.round(4))
        k_de, ly = P.de_xuat_k(TCK, KQPL)
        print(f"số PC đề xuất bằng thực nghiệm: {k_de} ({ly}); geoportal hiển thị K_PCA = {K_PCA}")
        HS5 = P.chot_k(HS, K_PCA, f"geoportal hiển thị {K_PCA} PC; đề xuất thực nghiệm k = {k_de}: {ly}")
        HS5.update(k_de_xuat=int(k_de), dac_trung=HS["dac_trung"])
        json.dump(HS5, open(p_hs, "w"))
    TROI = LP.troi_theo_cam_bien(MAU, HS5)
    TROI.to_csv(f"{KQ}/{PHUONG_AN}/troi_pc_theo_nam.csv", index=False)
    display(TROI.round(3))
    KG_PC = LP.keo_gian_pc(MAU, HS5)
    for y, t in sorted(NORM.items()):
        p_pc = f"{HF_LS}/lspc/lspc_{y}.tif"
        xam = [f"{HF_LS}/lspc{q}/lspc{q}_{y}.tif" for q in range(1, K_PCA + 1)]
        if not LAM_LAI and H.hop_le(p_pc) and all(H.hop_le(x) for x in xam):
            continue
        tmp = f"{TMP}/lspc_{y}.tif"
        LP.ap_anh(t, tmp, HS5)
        LS.lam_cog(tmp, p_pc)
        for q, x in enumerate(xam, 1):
            H.gray_cog(tmp, q, KG_PC[q][0], KG_PC[q][1], x, G3857, nodata=LS.NODATA, tmp_dir=TMP, verbose=False)
        os.remove(tmp)
    json.dump(LP.muc_manifest_lspc(HS5, NORM, KG_PC), open(f"{HF_LS}/lspc/he_so_lspc.json", "w"), ensure_ascii=False)
    print("PCA xong:", len(glob.glob(f"{HF_LS}/lspc/lspc_*.tif")), "năm")

In [ ]:
# ======================= 8. EMBEDDING LANDSAT (g7, 6 băng, GPU) =======================
EMB_XONG = False
if LAM["emb"]:
    import torch
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    if DEV == "cpu":
        print("KHÔNG có GPU: bỏ embedding (đổi runtime sang GPU rồi chạy lại ô này)")
    else:
        LE.self_test(DEV)
        ecfg = dict(LE.CAU_HINH_EMB, **EMB_CFG)
        os.makedirs(f"{KQ}/{PHUONG_AN}/emb", exist_ok=True)
        p_ck = f"{KQ}/{PHUONG_AN}/emb/{ecfg['ten']}.pt"
        MOM = LE.mo_men_chung(NORM)
        PJS2, TT = None, None
        if ecfg["chung_s2"]:
            from huggingface_hub import hf_hub_download
            man_hf = json.load(open(hf_hub_download(HF_REPO, "manifest.json", repo_type="dataset", force_download=True)))
            g = LE.tim_lop_g7(man_hf)
            assert g, "manifest chưa có embedding g7 của S2: đặt EMB_CFG chung_s2=False"
            g1, g2, p_pj = g
            PJS2 = json.load(open(hf_hub_download(HF_REPO, p_pj, repo_type="dataset")))
            url = lambda l, y: f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/" + l["duong_dan"].replace("{y}", str(y))
            TT = {y: LE.muc_tieu_s2(url(g1, y), url(g2, y), PJS2, L).astype(np.float16)
                  for y in ecfg["nam_s2"] if y in NORM and y in g1.get("nam", [])}
            print("mục tiêu S2:", sorted(TT))
        if os.path.exists(p_ck) and not LAM_LAI:
            ENC, ST = LE.nap_ckpt(p_ck, DEV)
            HEAD, TM_TS = ST.get("head"), ST.get("them", {})
        else:
            X, META, T = LE.cat_manh(NORM, ecfg["manh_moi_nam"], ecfg["manh"], seed=ecfg["seed"], mask=MASK, them=TT)
            Tn, TM_TS = None, {}
            if T is not None:
                Tn, tm, ts = LE.chuan_muc_tieu(T); TM_TS = dict(tm=tm, ts=ts)
            print(f"mảnh huấn luyện: {X.shape}, {META.nam.nunique()} năm")
            ENC = LE.tao_encoder(MOM, ecfg, DEV)
            HIST, HEAD = LE.huan_luyen(ENC, X, ecfg, T=Tn)
            LE.luu_ckpt(ENC, p_ck, ecfg, MOM, HEAD, them=TM_TS)
            del X, T, Tn
        # ---- đánh giá: R² đoán lại phổ và chỉ số từ embedding; ổn định giữa các năm, nhất là ở bước đổi cảm biến
        rr, cc, _ = LP.chon_diem(NORM[LP._nam_day_du(NORM)], 3000, 7, MASK)
        Z, X6, YR = LE.mau_z(ENC, NORM, rr_cc=(rr, cc))
        F9 = LP.dac_trung(X6.T / 1e4).T
        print("R² (hồi quy tuyến tính, phần để riêng) từ embedding:",
              dict(zip(LP.DAC_TRUNG, np.round(LE.r2_tuyen_tinh(Z, F9), 3))))
        Zn = {int(y): Z[YR == y] for y in np.unique(YR)}
        Xn = {int(y): (X6[YR == y] - np.asarray(MOM[0])) / np.asarray(MOM[1]) for y in np.unique(YR)}
        OD = LE.on_dinh_theo_nam(Zn, Xn, sorted(Zn))
        OD.to_csv(f"{KQ}/{PHUONG_AN}/emb/on_dinh_theo_nam.csv", index=False); display(OD.round(3))
        xau = OD[OD.chuyen_cam_bien & (OD.ti_so_voi_nen > 2)]
        if len(xau):
            print("CHÚ Ý: embedding trôi mạnh ở bước đổi cảm biến:", xau.cap.tolist(), "-> xem lại chuẩn hoá các năm đó")
        # ---- phép chiếu chung 64 -> 6 cho mọi năm
        if HEAD is not None and PJS2 is not None:
            A_, b_ = LE.dau_ra_ab(HEAD)
            PJ = LE.dau_ra_thanh_phep_chieu(A_, b_, TM_TS["tm"], TM_TS["ts"], PJS2["lo"], PJS2["hi"])
        else:
            PJ = H.fit_proj(Z[np.isfinite(Z).all(1)], n=ecfg["n_comp"])
        PJ.update(encoder=ecfg["ten"], chuan_hoa="chung mọi năm", mom=MOM, phuong_an=PHUONG_AN)
        os.makedirs(f"{HF_LS}/emb", exist_ok=True)
        json.dump(PJ, open(f"{HF_LS}/emb/lsg_phep_chieu.json", "w"))
        shutil.copy(p_ck, f"{HF_LS}/emb/{ecfg['ten']}.pt")
        for y, t in sorted(NORM.items()):
            o1, o2 = f"{HF_LS}/lsg/lsg_{y}.tif", f"{HF_LS}/lsgb/lsgb_{y}.tif"
            if not LAM_LAI and H.hop_le(o1) and H.hop_le(o2):
                continue
            q = f"{TMP}/lsq_{y}.tif"
            LE.ghi_chieu(ENC, t, q, PJ)
            H.rgb_cog_nhom(q, [([1, 2, 3], o1), ([4, 5, 6], o2)], G3857, tmp_dir=TMP, verbose=False)
            os.remove(q)
        EMB_XONG = True
        print("embedding xong:", len(glob.glob(f"{HF_LS}/lsg/lsg_*.tif")), "năm")

In [ ]:
# ======================= 9. MANIFEST VÀ ĐẨY LÊN HUGGING FACE =======================
NAM_CO = sorted(NORM)
tom_ch = {str(r.nam): dict(ap_dung=bool(r.ap_dung), n_pif=int(r.n_pif), r2_min=float(r.r2_min)) for r in BCH.itertuples()}
tom_cn = (BCN.groupby("bang")[["a", "b", "r2", "rmse"]].median().round(4).to_dict("index") if CN else {})
EMB_XONG = EMB_XONG or (all(os.path.exists(f"{HF_LS}/lsg/lsg_{y}.tif") for y in NAM_CO)
                        and os.path.exists(f"{HF_LS}/emb/lsg_phep_chieu.json"))
MUC_LS = LS.muc_manifest(NAM_CO, KG, L, PHUONG_AN, chuan_hoa=tom_ch, cau_noi=tom_cn,
                         ghi_chu="tham chiếu chuẩn hoá: " + os.path.basename(TC))
LOP = LS.lop_manifest(NAM_CO, co_pca=K_PCA if HS5 else 0, pca_kg=KG_PC, co_emb=EMB_XONG)
MUC_LSPC = LP.muc_manifest_lspc(HS5, NAM_CO, KG_PC) if HS5 else None
json.dump(dict(ls=MUC_LS, layers=LOP, lspc=MUC_LSPC), open(f"{KQ}/{PHUONG_AN}/manifest_ls_xem_truoc.json", "w"),
          ensure_ascii=False, indent=1)
print("lớp sẽ thêm:", [l["id"] for l in LOP], "| năm:", NAM_CO[0], "..", NAM_CO[-1], f"({len(NAM_CO)})")
if GUI_LEN and LAM["manifest"]:
    from huggingface_hub import HfApi
    try:
        from google.colab import userdata
        TOKEN = userdata.get("HF_TOKEN")             # Colab: biểu tượng chìa khoá (Secrets), tên HF_TOKEN
    except Exception:                                # noqa: BLE001
        from getpass import getpass
        TOKEN = getpass("Mã truy cập Hugging Face (quyền write): ")
    api = HfApi(token=TOKEN)
    api.upload_large_folder(folder_path=HF_LS, repo_id=HF_REPO, repo_type="dataset",
                            allow_patterns=["ls/*", "lstc/*", "lspc/*", "lspc[0-9]/*", "lsg/*", "lsgb/*", "emb/lsg*"])
    MOI = LS.cap_nhat_manifest_hf(api, HF_REPO, lambda man: LS.gop_manifest(man, MUC_LS, LOP, MUC_LSPC), tmp_dir=TMP)
    print("manifest: đã thêm", [l["id"] for l in MOI["layers"] if l.get("nguon") == "ls"], "| bản sao cũ ở manifest_luu/")
    del TOKEN

## Đọc kết quả
- `HP_LS_KETQUA/<A|B>/chuan_hoa.csv`: từng năm qua cổng IR-MAD hay không, số điểm ảnh bất biến, r² nhỏ nhất, hệ số đỏ và cận hồng ngoại.
- `cau_noi_s2.csv`: Landsat so với S2 cùng năm (hệ số gần 1, chặn gần 0 là tốt). Chỉ chẩn đoán, không sửa ảnh.
- `troi_pc_theo_nam.csv`: trung bình PC trên cùng bộ điểm qua các năm; cột `buoc_nhay_PCj` lớn bất thường ở 1999, 2013, 2022 là dấu hiệu lệch cảm biến còn sót.
- `emb/on_dinh_theo_nam.csv`: độ trôi embedding trên điểm có phổ ổn định; `ti_so_voi_nen` > 2 ở bước đổi cảm biến cần xem lại.
- Lớp mới trên geoportal (bản 3.0): `lstc` (màu thật), `lspc1..5`, `lsg`, `lsgb`; dữ liệu 10 băng `ls/ls_{năm}.tif` cho giá trị tại điểm và đường năm.